# 实验三 · 规约算子 ReduceSum —— 核内规约与核间规约

**所属**：《并行计算》第六章 · 昇腾 Ascend C 算子开发　|　**难度**：⭐⭐⭐⭐ 难点　|　**预计时长**：50–60 分钟

实验二的向量加法满足一条很强的性质：第 i 个输出只依赖第 i 个输入，各核之间不需要交换任何数据。本实验转向另一类负载——**规约**：输出只有一个标量，却依赖全部输入，因而各核算出的部分结果**必须合并**。这使实验二刻意回避的问题浮现出来：核间的数据交换。四个版本沿两条线索展开，先解决核内如何规约，再解决核间如何合并。

> **实验说明**
> 1. 本实验的核心内容有四点：`ReduceSum` 接口与 `TBuf` 临时缓冲、向量累加器、基于 workspace 的两阶段合并、以及 `SetAtomicAdd` 原子累加。前两点属于核内规约，后两点属于核间规约。
> 2. 本实验采用**递进式的版本组织**：以单核逐块规约为基准，每个版本仅引入一个新概念。
> 3. 请自上而下依次执行各单元格（Shift+Enter）。
> 4. 本实验依赖 **CANN 9.0.0 及以上**与 **Atlas A2/A3 训练推理系列产品**。
> 5. 四个版本的核函数、CPU 基准、数据生成、结果校验与 `main` 都写在同一个 `.asc` 文件中，由一条 `bisheng` 命令编译为单个可执行程序。
> 6. 元素总数与随机种子都是**运行时参数**，规模扫描与精度分析不需要重新编译。
> 7. 本实验的**精度判定口径与前两个实验不同**：规约改变了浮点数的累加顺序，容差须相应放宽。放宽到多少，由 §5 的误差分析决定，而不是「能通过为止」。
> 8. 本实验建立在**实验二 向量加法**的基础之上，建议先完成实验二。

## 🎯 学习目标

完成本实验后，开发者应能够：

- 说明规约运算为何比逐元素运算更难并行，指出其数据依赖的来源
- 掌握 `ReduceSum` 接口的用法，能核算它所需的临时缓冲大小
- 说明官方归约 API 家族（`Reduce*` / `WholeReduce*` / `BlockReduce*` / `PairReduceSum`）各自的归约范围，并说明官方在大数据量场景下推荐的方案次序
- 说明多核同时写入同一段 Global Memory 为何会被硬件串行化，以及 512 字节这条粒度的来源
- 掌握 `TBuf` 与 `TQue` 的区别，说明何时应当使用 `VECCALC` 位置的临时缓冲
- 理解**向量累加器**：部分和保留在向量中，末尾只做一次规约
- 掌握**两级规约**结构，说明核内规约与核间规约各自受什么约束
- 掌握基于 workspace 的两阶段合并方法，说明为何两次核函数启动即可保证阶段顺序
- 掌握 `SetAtomicAdd` / `SetAtomicNone` 的用法，说明启动前必须清零输出的原因
- 能定量比较两种核间合并方式，并解释其结论与第五章直方图实验的关系
- 能用「串行累加链的长度」同时解释四个版本的性能差异与精度差异
- 能依据误差的增长规律选择容差，而不是依据「能否通过」

## 🗺️ 学习路径

1. **准备阶段**：理解规约的数据依赖，以及两级规约结构
2. **知识铺垫**：核内规约的两种写法、核间合并的两条路线、串行累加链这条主线
3. **Device 侧实现**：三个算子类覆盖四个版本
4. **Host 侧实现**：数据生成、float64 参考真值、CPU 基准与统一计时
5. **编译运行**：一条 `bisheng` 命令，一次运行产出全部对照数据
6. **结果可视化**：以 CPU 与 v1 两套基线分别计算加速比
7. **参数扫描**：核数扫描比较两种核间合并方式，规模扫描定位合并开销的作用区间
8. **精度分析**：验证误差与串行累加链长度的关系
9. **结果分析**：把性能与精度的结论收束到同一条主线上

## 1. 背景与动机：规约为什么更难

实验二的向量加法满足一条很强的性质：

$$ z_i = f(x_i, y_i) $$

第 i 个输出只依赖第 i 个输入。把数据切成任意份、交给任意多个核，各核的输出互不重叠，合并时只需把各段拼在一起——事实上根本不需要合并这个动作。

规约不同：

$$ s = \sum_{i=0}^{N-1} x_i $$

**唯一的输出依赖全部输入。** 无论怎样切分，各核只能算出一个**部分和**，这些部分和必须再合并一次才能得到最终结果。这一步合并跨越了核的边界，而各核的片上缓冲彼此独立，因此必须借助 Global Memory 完成。

<!-- markdown 版本（保留备用；如需切回，删除本注释标记并注释掉下方 HTML 表格）
| 对比项 | 逐元素（实验二） | 规约（本实验） |
| --- | --- | --- |
| 输出与输入的对应 | 一一对应 | 多对一 |
| 核间是否需要交换数据 | 不需要 | **必须** |
| 输出规模 | 与输入同量级 | 单个标量 |
| 浮点结果是否依赖计算顺序 | 否 | **是** |
-->

<table style="margin-left:0; margin-right:auto; border-collapse:collapse; text-align:left;">
<thead>
<tr>
<th style="border:1px solid #cccccc; padding:6px 12px; text-align:left; background-color:#f5f5f5;">对比项</th>
<th style="border:1px solid #cccccc; padding:6px 12px; text-align:left; background-color:#f5f5f5;">逐元素（实验二）</th>
<th style="border:1px solid #cccccc; padding:6px 12px; text-align:left; background-color:#f5f5f5;">规约（本实验）</th>
</tr>
</thead>
<tbody>
<tr>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">输出与输入的对应</td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">一一对应</td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">多对一</td>
</tr>
<tr>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">核间是否需要交换数据</td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">不需要</td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;"><strong>必须</strong></td>
</tr>
<tr>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">输出规模</td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">与输入同量级</td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">单个标量</td>
</tr>
<tr>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">浮点结果是否依赖计算顺序</td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">否</td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;"><strong>是</strong></td>
</tr>
</tbody>
</table>

最后一行值得特别注意：浮点加法不满足结合律，改变累加顺序会得到略有差异的结果。并行规约必然改变累加顺序，因此**并行版本与串行版本的结果不可能逐位相同**。这不是缺陷，而是必须正确对待的事实——§5 会进一步说明，改变累加顺序既可能让结果变差，也可能让结果变好。

### 算法与数据规格

$$ s = \sum_{i=0}^{N-1} x_i, \quad N = 2^{21} = 2\,097\,152 $$

<!-- markdown 版本（保留备用；如需切回，删除本注释标记并注释掉下方 HTML 表格）
| 项目 | 取值 | 说明 |
| --- | --- | --- |
| 数据类型 | `float`（FP32） | 与前两个实验一致 |
| 元素总数 N | 2^21 = 2 097 152 | **运行时参数**，可由命令行覆盖 |
| 输入数据分布 | [0, 1) 上的均匀分布 | 求和结果约为 N/2 ≈ 1.05×10^6 |
| 分块长度 TILE_LENGTH | 4096 个元素 | 编译期常量，可由 `-D` 覆盖 |
| 参与计算的核数 | 8 | 编译期常量，可由 `-D` 覆盖 |
| 参考真值 | Host 侧 `float64` 顺序累加 | 参考值的精度必须高于被测对象 |
-->

<table style="margin-left:0; margin-right:auto; border-collapse:collapse; text-align:left;">
<thead>
<tr>
<th style="border:1px solid #cccccc; padding:6px 12px; text-align:left; background-color:#f5f5f5;">项目</th>
<th style="border:1px solid #cccccc; padding:6px 12px; text-align:left; background-color:#f5f5f5;">取值</th>
<th style="border:1px solid #cccccc; padding:6px 12px; text-align:left; background-color:#f5f5f5;">说明</th>
</tr>
</thead>
<tbody>
<tr>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">数据类型</td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;"><code>float</code>（FP32）</td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">与前两个实验一致</td>
</tr>
<tr>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">元素总数 N</td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">2^21 = 2 097 152</td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;"><strong>运行时参数</strong>，可由命令行覆盖</td>
</tr>
<tr>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">输入数据分布</td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">[0, 1) 上的均匀分布</td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">求和结果约为 N/2 ≈ 1.05×10^6</td>
</tr>
<tr>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">分块长度 TILE_LENGTH</td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">4096 个元素</td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">编译期常量，可由 <code>-D</code> 覆盖</td>
</tr>
<tr>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">参与计算的核数</td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">8</td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">编译期常量，可由 <code>-D</code> 覆盖</td>
</tr>
<tr>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">参考真值</td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">Host 侧 <code>float64</code> 顺序累加</td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">参考值的精度必须高于被测对象</td>
</tr>
</tbody>
</table>

**为什么输入取 [0, 1) 而不是 [−1, 1)。** 若取 [−1, 1)，求和结果会落在 0 附近，此时相对误差的分母趋于零，判据随之失去意义。数据分布的选取本身就是实验设计的一部分。

## 2. 两级规约结构

规约的实现必然分为两级，二者的约束完全不同：

<img src="images/06.03_two_level_reduce.png" alt="两级规约结构" width="800">

<!-- markdown 版本（保留备用；如需切回，删除本注释标记并注释掉下方 HTML 表格）
| 级别 | 参与者 | 通信介质 | 主要难点 |
| --- | --- | --- | --- |
| 核内规约 | 单个核内的矢量单元 | 片上缓冲 | 规约指令的调用次数与临时空间 |
| 核间规约 | 全部参与计算的核 | **Global Memory** | 跨核的数据交换与阶段顺序 |
-->

<table style="margin-left:0; margin-right:auto; border-collapse:collapse; text-align:left;">
<thead>
<tr>
<th style="border:1px solid #cccccc; padding:6px 12px; text-align:left; background-color:#f5f5f5;">级别</th>
<th style="border:1px solid #cccccc; padding:6px 12px; text-align:left; background-color:#f5f5f5;">参与者</th>
<th style="border:1px solid #cccccc; padding:6px 12px; text-align:left; background-color:#f5f5f5;">通信介质</th>
<th style="border:1px solid #cccccc; padding:6px 12px; text-align:left; background-color:#f5f5f5;">主要难点</th>
</tr>
</thead>
<tbody>
<tr>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">核内规约</td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">单个核内的矢量单元</td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">片上缓冲</td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">规约指令的调用次数与临时空间</td>
</tr>
<tr>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">核间规约</td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">全部参与计算的核</td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;"><strong>Global Memory</strong></td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">跨核的数据交换与阶段顺序</td>
</tr>
</tbody>
</table>

本实验的 v1 与 v2 讨论第一级，v3 与 v4 讨论第二级。

### 2.1 核内规约：ReduceSum 接口

昇腾提供的归约 API 按**归约的数据范围**分为四类，官方文档以下图作对照：

<img src="images/06.03_reduce_instructions.png" alt="06.03_reduce_instructions" width="760px">

<!-- markdown 版本（保留备用；如需切回，删除本注释标记并注释掉下方 HTML 表格）
| API 家族 | 归约范围 | 输出 |
| --- | --- | --- |
| `ReduceMax` / `ReduceMin` / `ReduceSum` | **全部输入数据** | 1 个结果 |
| `WholeReduceMax` / `WholeReduceMin` / `WholeReduceSum` | 每个 repeat 内的数据 | 每 repeat 1 个结果 |
| `BlockReduceMax` / `BlockReduceMin` / `BlockReduceSum` | 每个 datablock 内的数据 | 每 datablock 1 个结果 |
| `PairReduceSum` | 相邻的奇偶两个元素 | 元素数减半 |
-->

<table style="margin-left:0; margin-right:auto; border-collapse:collapse; text-align:left;">
<thead>
<tr>
<th style="border:1px solid #cccccc; padding:6px 12px; text-align:left; background-color:#f5f5f5;">API 家族</th>
<th style="border:1px solid #cccccc; padding:6px 12px; text-align:left; background-color:#f5f5f5;">归约范围</th>
<th style="border:1px solid #cccccc; padding:6px 12px; text-align:left; background-color:#f5f5f5;">输出</th>
</tr>
</thead>
<tbody>
<tr>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;"><code>ReduceMax</code> / <code>ReduceMin</code> / <code>ReduceSum</code></td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;"><strong>全部输入数据</strong></td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">1 个结果</td>
</tr>
<tr>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;"><code>WholeReduceMax</code> / <code>WholeReduceMin</code> / <code>WholeReduceSum</code></td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">每个 repeat 内的数据</td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">每 repeat 1 个结果</td>
</tr>
<tr>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;"><code>BlockReduceMax</code> / <code>BlockReduceMin</code> / <code>BlockReduceSum</code></td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">每个 datablock 内的数据</td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">每 datablock 1 个结果</td>
</tr>
<tr>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;"><code>PairReduceSum</code></td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">相邻的奇偶两个元素</td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">元素数减半</td>
</tr>
</tbody>
</table>

本实验取第一类中的 `ReduceSum`，它一次调用即可把整段数据归约为一个值，语义最直接：

```cpp
AscendC::ReduceSum(dstLocal, srcLocal, tmpLocal, count);
```

该接口把 `srcLocal` 的前 `count` 个元素求和，结果写入 `dstLocal` 的第 0 个元素。官方对这一族接口另有两点说明：其一，它**由多种指令组合实现**，而不是单条硬件指令；其二，正因为如此，它对 repeat 次数超过 255 的情形在接口内部作了处理，调用方不必自行拆分。第一点关系到性能，§3.3 会接着讨论。

**为什么需要第三个参数 `tmpLocal`。** 规约不是一步完成的。硬件按 repeat 分批处理，第一轮先得到每个 repeat 的部分和，这些中间结果需要一块暂存空间才能继续规约。该空间由开发者提供，接口本身不申请内存。所需元素数按下式核算：

```text
elementsPerRepeat = 256 / sizeof(T)                 # float 为 64
firstMaxRepeat    = count / elementsPerRepeat       # 4096 / 64 = 64
tmpElements       = RoundUp(firstMaxRepeat, 8) * 8  # 64 -> 512
```

`count = 4096`、`T = float` 时算得 512 个元素。代码中直接按该式定义 `TMP_LENGTH`，修改 `TILE_LENGTH` 时会自动跟随，不需要手工重算。

### 2.2 TBuf 与 TQue 的区别

至此出现了第二类片上缓冲：

<!-- markdown 版本（保留备用；如需切回，删除本注释标记并注释掉下方 HTML 表格）
| 类型 | 位置 | 用途 | 是否参与流水同步 |
| --- | --- | --- | --- |
| `TQue` | `VECIN` / `VECOUT` | 在流水任务之间传递数据 | 是，`EnQue` / `DeQue` 携带同步语义 |
| `TBuf` | `VECCALC` | 核内计算的临时空间与累加器 | 否，仅通过 `Get` 取用 |
-->

<table style="margin-left:0; margin-right:auto; border-collapse:collapse; text-align:left;">
<thead>
<tr>
<th style="border:1px solid #cccccc; padding:6px 12px; text-align:left; background-color:#f5f5f5;">类型</th>
<th style="border:1px solid #cccccc; padding:6px 12px; text-align:left; background-color:#f5f5f5;">位置</th>
<th style="border:1px solid #cccccc; padding:6px 12px; text-align:left; background-color:#f5f5f5;">用途</th>
<th style="border:1px solid #cccccc; padding:6px 12px; text-align:left; background-color:#f5f5f5;">是否参与流水同步</th>
</tr>
</thead>
<tbody>
<tr>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;"><code>TQue</code></td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;"><code>VECIN</code> / <code>VECOUT</code></td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">在流水任务之间传递数据</td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">是，<code>EnQue</code> / <code>DeQue</code> 携带同步语义</td>
</tr>
<tr>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;"><code>TBuf</code></td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;"><code>VECCALC</code></td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">核内计算的临时空间与累加器</td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">否，仅通过 <code>Get</code> 取用</td>
</tr>
</tbody>
</table>

判断准则很简单：**数据是否需要跨流水任务传递**。`ReduceSum` 的临时空间只在 `Compute` 内部使用，累加器也只被 `Compute` 读写，二者都不跨任务，因此都用 `TBuf`。

`TBuf` 的 `InitBuffer` 只有两个参数——没有缓冲块数，因为它不参与流水同步；取用时调用 `Get<T>()` 而不是 `AllocTensor`，用完也不需要 `FreeTensor`。官方对二者关系的表述是：`TBuf` 申请的空间**只能参与计算，无法执行队列的入队出队操作**。

<img src="images/06.03_memory_management.png" alt="06.03_memory_management" width="820px">

*本图只画出了队列这一条路径：`InitBuffer` 由 `TPipe` 从片上存储中划出缓冲，`AllocTensor` 与 `FreeTensor` 在缓冲与 `LocalTensor` 之间收发。`TBuf` 走的是另一条更短的路径——同样由 `InitBuffer` 划出空间，但取用时直接 `Get`，不经过入队与出队*

## 3. 核内规约的两种写法

<img src="images/06.03_vector_accumulator.png" alt="逐块规约与向量累加器的对照" width="800">

### 3.1 v1：逐块规约

最直接的写法是每搬入一个分块就规约一次，得到一个标量，再累加到总和上：

```cpp
AscendC::ReduceSum(reduceDst, xLocal, tmpBuf, TILE_LENGTH);
sum_ += reduceDst.GetValue(0);        // Scalar 读取片上缓冲
```

规约指令的执行次数等于分块数，N = 2^21 时为 512 次。

**关于 `GetValue`。** 它属于 Scalar 流水操作，与前一条矢量指令之间存在数据依赖。在算子工程使能自动同步的情况下，开发者无须手工插入同步事件；若关闭自动同步，则需自行插入 `SetFlag` / `WaitFlag`。这一点在思考题中还会再次讨论。

### 3.2 v2：向量累加器

更好的写法是让部分和**保留在向量中**：内层循环只做逐元素累加，直到全部分块处理完毕，才执行**唯一一次**规约。

```cpp
AscendC::Add(acc, acc, xLocal, TILE_LENGTH);   // 循环内：4096 路并行累加
// ...
AscendC::ReduceSum(reduceDst, acc, tmpBuf, TILE_LENGTH);   // 循环外：只做一次
```

**与第三章的对照**：这正是 GEMV 实验中「内层循环全程让部分和保留在向量寄存器中累加，只在每行末尾执行一次水平规约」的同一思想。区别仅在于向量的宽度：NEON 的一条向量寄存器容纳 4 个 float，而此处的累加器有 4096 个元素。

**累加器的清零时机**是本版最容易出错的地方：必须在进入循环之前完成一次，且**只能做一次**。若误置于循环内部，每轮都会把此前累积的部分和清掉，结果将只剩最后一个分块的和。动手练习第 1 题要求实际制造一次这个错误。

### 3.3 官方在大数据量场景下推荐的方案：二分累加

上面两种写法解决的是**规约调用多少次**。至于**每一次规约本身怎么做得更快**，官方给出了另一条路线。

官方的实测结论是：`WholeReduceSum` 等归约指令的延迟约为 `Add` 指令的 **2 至 5 倍**。因此对连续数据的归约，可以先反复用 `Add` 把数据折半相加，待剩余数据量不超过一个 repeat（256 字节）时，再用一条 `WholeReduceSum` 收尾。该方案称为**二分累加**。

<img src="images/06.03_binary_reduce.png" alt="06.03_binary_reduce" width="900px">

官方给出的性能次序是：**数据量较大、循环次数较多**的场景下，二分累加方案 > `WholeReduceSum` 单指令 > `ReduceSum` 接口；小数据量或特殊 shape 下则需具体分析。官方的对照数据为：输入 shape 为 30000 的 `float` 数据，二分累加 172 cycle，`WholeReduceSum` 单指令 242 cycle。进一步地，`BlockReduceSum` 的执行效率优于 `WholeReduceSum`，二者组合可以得到更好的结果。完整样例见官方的 `ReduceCustom`。

**那么本实验为什么仍然使用 `ReduceSum`？** 两个理由：

1. **教学次序**。`ReduceSum` 一次调用即完成整段归约，不需要掌握 mask 计数模式、`SetVectorMask`、`PipeBarrier` 等一整套细节，适合作为规约的入门接口。二分累加方案的官方样例代码需要用到上述全部内容，留到动手练习第 8 题。
2. **它在本实验中不是主导项**。官方的性能次序针对的正是「循环次数较多」的场景——这恰好是 v1 的情形，v1 要调用 512 次 `ReduceSum`，接口选择的代价在此被放大 512 倍。而 v2 之后规约只执行一次，其耗时相对整段数据的搬入与累加可以忽略，此时换用哪种归约方案都不会改变结论。

换句话说，**v1 慢的原因有两层**：规约调用了 512 次，且每次调用用的是官方性能次序中最靠后的那个接口。v2 消除的是第一层，二分累加针对的是第二层。

## 4. 核间合并的两条路线

各核的片上缓冲彼此独立，核之间没有共享的片上存储，因此部分和**无法直接相加**。本实验给出两条路线。

### 4.1 路线一：workspace 两阶段合并（v3）

解决办法是借道 Global Memory：在设备上额外申请一块内存，称为 **workspace**。它不是算子的输入，也不是输出，而是算子内部使用的暂存区。官方把「算子内部需要额外的设备内存做数据交换」列为使用用户 workspace 的典型场景之一，核间规约正属于这一类。

> **两种开发方式下 workspace 的获取方式不同。** 本实验采用的是直调工程：workspace 由 Host 侧用 `aclrtMalloc` 申请，作为一个普通的 `GM_ADDR` 形参传给核函数，与输入输出没有区别。而在算子工程与框架接入的场景下，官方提供了专门的机制——编译时加上 `HAVE_WORKSPACE` 宏，框架会把核函数的指定形参识别为 workspace 并自动完成系统区域的偏移，Tiling 阶段则通过 `GetWorkspaceSizes` 申报所需大小。两者的作用相同，接口形态不同。

<img src="images/06.03_workspace.png" alt="workspace 在算子内部的作用" width="900">

```text
workspace 布局（BLOCK_DIM = 8）：

 核0        核1        核2                核7
[s0,0×7]  [s1,0×7]  [s2,0×7]  ......  [s7,0×7]
 └ 8 个元素 ┘
```

每个核占用 8 个元素（32 字节，满足对齐要求），其中第 0 个存放该核的部分和，其余 7 个置零。第二阶段只需把这 64 个元素全部相加即可——填充的零不影响结果。**这一布局把对齐约束转化成了一个不必特殊处理的实现细节。**

**但这一布局还有一个不那么显然的后果。** 官方文档指出：各计算单元访问 Global Memory 时，地址请求按 **512 字节**粒度对齐后处理；当多个核同时访问的地址落在**连续的 512 字节范围内**时，出于数据一致性的原因，这些请求会被**硬件串行处理**，并且同地址访问的核数越多，串行导致的性能劣化越严重。

<img src="images/06.03_same_address_serialize.png" alt="06.03_same_address_serialize" width="820px">

*图中 addr0～addr5 是六个不同的地址，但因落在同一个连续的 512 字节范围内，被视为同一个地址请求而串行处理*

官方同时说明：算子执行机制保证核函数入参（含 workspace）的地址按 512 字节对齐，因此只需按偏移量即可判断两个地址是否落入同一窗口。按上面的布局，8 个核的 workspace 合计只有 8 × 32 = 256 字节，**整体落在一个 512 字节窗口之内**；核数增大到 16 时恰好占满一个窗口，仍然如此。也就是说，第一阶段各核写 workspace 这一步是被串行化的。官方给出的规避办法是**修改切分策略**——把每个核的槽位撑开到 512 字节的整数倍，使各核落入不同的窗口。本实验保留 32 字节的紧凑布局，因为它更能说明对齐约束本身；把槽位撑开到 512 字节并重新测量，留作动手练习第 6 题。

这一条同时是理解 §15 ③ 的关键：v3 与 v4 都要向一小段 Global Memory 集中写入，**两者在这一点上受同样的约束**。

**为什么两次核函数启动就够了。** 第二阶段必须在**全部**第一阶段的核完成写入之后才能开始。同一 Stream 内的任务按下发顺序串行执行，因此第二个核函数必然在第一个全部完成之后才启动，不需要额外的同步代码。

另一种做法是在单次启动内使用核间同步接口 `SyncAll`，让所有核在写完 workspace 后集体等待，再由 0 号核完成合并。采用这条路线时有一条硬性约束需要留意：官方明确要求**逻辑核数 `NumBlocks` 不得大于实际运行该算子的 AI 处理器核数**，否则框架插入的同步会出现异常，导致核函数卡死。两次启动的写法不受这条约束——这也是二者取舍的一部分，详见思考题第 3 题与动手练习第 7 题。

核间同步的一般形态如下图：两个核经由同一块 Global Memory 传递数据时，写入方置标记、读取方等待标记，先后顺序才能确定。

<img src="images/06.03_cross_core_sync.png" alt="06.03_cross_core_sync" width="760px">

*图中 `CrossCoreSetFlag` 与 `CrossCoreWaitFlag` 配对使用，标记 ID 对应一个计数器；`SyncAll` 是在此之上封装的全核集合同步*

### 4.2 路线二：原子累加（v4）

v3 的第二阶段做的事情很少——只把 64 个元素相加——却付出了一次完整的核函数启动与一趟 workspace 的写入读出。v4 把这一步交给硬件完成：

```cpp
AscendC::SetAtomicAdd<LabDType>();              // 开启原子累加
AscendC::DataCopy(zGm, outLocal, OUT_LENGTH);   // 本次搬出由覆盖改为累加
AscendC::SetAtomicNone();                       // 关闭，避免影响后续指令
```

开启后，`DataCopy` 写向 Global Memory 的行为由覆盖变为累加。8 个核各自把 `[s_k, 0, 0, ..., 0]` 累加到同一段 32 字节上，最终 `z[0]` 即为 8 个部分和之和。

> **接口名随产品型号而变。** 关闭原子累加的接口，在 Atlas A2 / A3 训练推理系列产品上是 `SetAtomicNone`；在 Atlas 350 加速卡上则改为 `DisableDmaAtomic`，且该产品的 API 清单中不再有 `SetAtomicNone`。阅读官方样例代码时需要先确认样例针对的产品型号。本实验运行在 A2 / A3 上，因此使用 `SetAtomicNone`。

多核把部分结果累加到同一块输出，是官方在矩阵乘**多核切 K** 场景中采用的同一种做法：M、N 方向切不动时改切 K 方向以提高并行度，各核算出的部分积用原子累加合并到同一块 C 矩阵上。

<img src="images/06.03_split_k_atomic.png" alt="06.03_split_k_atomic" width="820px">

*开启多核切 K。A1/A2/A3 与 B1/B2/B3 的部分积累加到同一块结果 R 上*

**两条必须遵守的约束**：

<!-- markdown 版本（保留备用；如需切回，删除本注释标记并注释掉下方 HTML 表格）
| 约束 | 原因 |
| --- | --- |
| **启动前必须清零输出** | 原子累加是在已有值上继续相加，硬件不会先清零。官方的表述是：如果不预先清零 Global Memory，可能会因为累加原始无效数据而产生精度问题 |
| **用完必须 `SetAtomicNone`** | 该设置会影响此后所有的搬出指令 |
-->

<table style="margin-left:0; margin-right:auto; border-collapse:collapse; text-align:left;">
<thead>
<tr>
<th style="border:1px solid #cccccc; padding:6px 12px; text-align:left; background-color:#f5f5f5;">约束</th>
<th style="border:1px solid #cccccc; padding:6px 12px; text-align:left; background-color:#f5f5f5;">原因</th>
</tr>
</thead>
<tbody>
<tr>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;"><strong>启动前必须清零输出</strong></td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">原子累加是在已有值上继续相加，硬件不会先清零。官方的表述是：如果不预先清零 Global Memory，可能会因为累加原始无效数据而产生精度问题</td>
</tr>
<tr>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">用完必须 <code>SetAtomicNone</code></td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">该设置会影响此后所有的搬出指令</td>
</tr>
</tbody>
</table>

第一条有一个直接的后果：**v4 的结果必须在计时循环之前取走**。计时循环会连续启动 `REPEAT` 次核函数，输出会被累加 `REPEAT` 次。本实验因此把每个版本的流程统一定为「先清零 → 运行一次并校验 → 再计时」。这也说明原子累加有一项隐性成本：**清零输出的责任被转移给了调用方**。至于省去一次核函数启动与一趟 workspace 往返实际能省下多少时间，要由实测回答。

### 4.3 与第五章直方图实验的对照

第五章的结论是：对高频操作使用原子指令代价很高，线程局部化后合并更优。本实验的情形恰好相反，原因在于**原子操作的次数**完全不同：

<!-- markdown 版本（保留备用；如需切回，删除本注释标记并注释掉下方 HTML 表格）
| 对比项 | 第五章 V2（`atomic`） | 本实验 v4 |
| --- | --- | --- |
| 原子操作次数 | N = 10^6 次 | `BLOCK_DIM` = 8 次 |
| 冲突概率 | 约 1/B，B 为 bin 数 | 8 个核争用同一地址，但只争一次 |
| 相对代价 | 极高 | 可忽略 |
-->

<table style="margin-left:0; margin-right:auto; border-collapse:collapse; text-align:left;">
<thead>
<tr>
<th style="border:1px solid #cccccc; padding:6px 12px; text-align:left; background-color:#f5f5f5;">对比项</th>
<th style="border:1px solid #cccccc; padding:6px 12px; text-align:left; background-color:#f5f5f5;">第五章 V2（<code>atomic</code>）</th>
<th style="border:1px solid #cccccc; padding:6px 12px; text-align:left; background-color:#f5f5f5;">本实验 v4</th>
</tr>
</thead>
<tbody>
<tr>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">原子操作次数</td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">N = 10^6 次</td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;"><code>BLOCK_DIM</code> = 8 次</td>
</tr>
<tr>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">冲突概率</td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">约 1/B，B 为 bin 数</td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">8 个核争用同一地址，但只争一次</td>
</tr>
<tr>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">相对代价</td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">极高</td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">可忽略</td>
</tr>
</tbody>
</table>

**两条结论并不矛盾，判据是同一个：原子操作的总次数。** 只要先在核内把数据规约成一个标量，把原子操作的次数压到与核数同量级，原子累加就是一种廉价而简洁的合并方式。这与第五章「先局部化、再合并」其实是同一条原则的两种表现。

## 5. 一条主线：串行累加链

本实验的四个版本在性能和精度上都有差异，而这两组差异指向同一个量：**最长的那条串行累加链有多长，以及有多少条这样的链在同时推进**。

<img src="images/06.03_accumulation_chain.png" alt="四种累加结构的串行链长与通道数" width="1000">

把一次求和看作若干条**独立通道**，每条通道内部是一串必须依次执行的加法。记单条通道的长度为 k、通道数为 L，则四种实现的对照如下（N = 2^21，TILE_LENGTH = 4096，BLOCK_DIM = 8）：

<!-- markdown 版本（保留备用；如需切回，删除本注释标记并注释掉下方 HTML 表格）
| 实现 | 单条通道的链长 k | 独立通道数 L | 规约指令次数 |
| --- | --- | --- | --- |
| CPU 串行 float32 | N = 2 097 152 | 1 | — |
| v1 逐块规约 | N / TILE_LENGTH = 512 | 1 | 512 |
| v2 向量累加器 | N / TILE_LENGTH = 512 | TILE_LENGTH = 4096 | 1 |
| v3 / v4 多核 + 向量累加器 | N / (BLOCK_DIM × TILE_LENGTH) = 64 | BLOCK_DIM × TILE_LENGTH = 32 768 | 每核 1 |
-->

<table style="margin-left:0; margin-right:auto; border-collapse:collapse; text-align:left;">
<thead>
<tr>
<th style="border:1px solid #cccccc; padding:6px 12px; text-align:left; background-color:#f5f5f5;">实现</th>
<th style="border:1px solid #cccccc; padding:6px 12px; text-align:left; background-color:#f5f5f5;">单条通道的链长 k</th>
<th style="border:1px solid #cccccc; padding:6px 12px; text-align:left; background-color:#f5f5f5;">独立通道数 L</th>
<th style="border:1px solid #cccccc; padding:6px 12px; text-align:left; background-color:#f5f5f5;">规约指令次数</th>
</tr>
</thead>
<tbody>
<tr>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">CPU 串行 float32</td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">N = 2 097 152</td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">1</td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">—</td>
</tr>
<tr>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">v1 逐块规约</td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">N / TILE_LENGTH = 512</td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">1</td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">512</td>
</tr>
<tr>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">v2 向量累加器</td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">N / TILE_LENGTH = 512</td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">TILE_LENGTH = 4096</td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">1</td>
</tr>
<tr>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">v3 / v4 多核 + 向量累加器</td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">N / (BLOCK_DIM × TILE_LENGTH) = 64</td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">BLOCK_DIM × TILE_LENGTH = 32 768</td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">每核 1</td>
</tr>
</tbody>
</table>

**对精度的影响。** 每一次浮点加法引入一次舍入，误差沿链累积；而不同通道的舍入误差彼此独立、符号随机，合并时会相互抵消。因此相对误差大致按

$$ \varepsilon \cdot \sqrt{k} \,/\, \sqrt{L} $$

变化，其中 ε 为 FP32 的机器精度（约 1.19×10^-7）。**链越长误差越大，通道越多误差越小。**这是一个**数量级估计**：它假定各次舍入的误差彼此独立、符号随机，因此只能用来判断趋势和量级，不能用来预测某一次运行的具体数值。它还有一条适用下界——当误差降到 float32 在该量级下的最小间隔（1 ULP，本实验中相对量约 6×10^-8）附近时，公式就不再有分辨力，§15 ⑥ 会用实测数据说明这一点。§14 会用实测数据检验这个趋势。

**对性能的影响。** 同一条链上的加法必须依次完成，无法重叠。CPU 基准的 2^21 次加法构成一条完整的依赖链；v1 每轮还要额外付出一次规约指令和一次矢量到标量的取值；v2 把 4096 条通道压进一条矢量指令；v3 与 v4 进一步把链长再除以核数。

由此得到本实验的一条判断：**缩短串行累加链，同时改善性能与精度。** 这与直觉相反——通常认为并行化是以精度为代价换取速度，但在规约这一类算子上，两者的方向是一致的。

> **需要说明的是**，性能上 v1 与 v2 的差距主要来自规约指令的执行次数和矢量与标量之间的同步，而不只是链长本身；链长这一概念在精度上的解释力更强。§15 会分别给出两者的证据。

## 6. 本实验的测量方法

性能测量沿用实验二 §5 的做法：两个指标（`cpu_ms`、`kernel_ms`）与四条要点（预热、多次重复取平均、先同步再停止计时、基准与被测使用相同的优化级别），此处不再复述。与实验二一致，本实验也不单独测量「主机拷贝 + 核函数 + 设备回传」这一端到端口径，理由见实验二 §5.1 与本实验 §15 ④。本节只说明本实验特有的四点。

**一、精度口径与前两个实验不同。**

<!-- markdown 版本（保留备用；如需切回，删除本注释标记并注释掉下方 HTML 表格）
| 项目 | 做法 | 理由 |
| --- | --- | --- |
| 参考真值 | Host 侧 `float64` 顺序累加 | 参考值的精度必须高于被测对象 |
| 判据 | 相对误差 | 输出只有一个标量，无法统计「超差元素数」 |
| 容差 | 1e-4 | 依据 §5 的误差规律选取，见下 |
-->

<table style="margin-left:0; margin-right:auto; border-collapse:collapse; text-align:left;">
<thead>
<tr>
<th style="border:1px solid #cccccc; padding:6px 12px; text-align:left; background-color:#f5f5f5;">项目</th>
<th style="border:1px solid #cccccc; padding:6px 12px; text-align:left; background-color:#f5f5f5;">做法</th>
<th style="border:1px solid #cccccc; padding:6px 12px; text-align:left; background-color:#f5f5f5;">理由</th>
</tr>
</thead>
<tbody>
<tr>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">参考真值</td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">Host 侧 <code>float64</code> 顺序累加</td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">参考值的精度必须高于被测对象</td>
</tr>
<tr>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">判据</td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">相对误差</td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">输出只有一个标量，无法统计「超差元素数」</td>
</tr>
<tr>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">容差</td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">1e-4</td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">依据 §5 的误差规律选取，见下</td>
</tr>
</tbody>
</table>

**二、容差为什么取 1e-4。** 按 §5 的规律，链最长的一端是 CPU 的串行 `float32` 累加，其相对误差量级为 ε·√N ≈ 1.19×10^-7 × 1448 ≈ 1.7×10^-4，实测通常比这个上界小一个量级。四个 NPU 版本的链都短得多，误差要小若干个数量级。取 1e-4 可以覆盖其中误差最大的实现，同时仍能拦住真正的算法错误——例如动手练习第 1 题制造的清零错误，其结果会与真值相差数百倍。

**三、标量输出更容易被编译器优化掉。** CPU 基准每一轮算出的都是同一个和，`-O2` 下编译器完全可以只算一次、把整个重复循环消掉，测得的耗时会接近于零。本实验让累加器的初值来自一个 `volatile` 变量：编译器无法证明各轮的初值相同，只能老实地重复执行。实验二只要求「基准实现的结果必须被使用」，那是因为它的输出是一整个数组；输出退化为一个标量时，仅仅使用结果还不够。

**四、v4 的结果必须在计时之前取走。** 原因见 §4.2。四个版本因此统一为「清零输出 → 运行一次并校验 → 再计时」的流程。

## 7. 环境准备与检查

In [ ]:
!mkdir -p src_reduce

import os, subprocess

env = subprocess.check_output(
    "bash -l -c 'source $ASCEND_TOOLKIT_HOME/set_env.sh && env'", shell=True, text=True
)
for line in env.splitlines():
    if "=" in line:
        os.environ.__setitem__(*line.split("=", 1))
print("🎉 环境变量导入完成")

In [ ]:
import shutil, subprocess

print("bisheng  :", shutil.which("bisheng") or "⚠️  未找到，请重新执行上一个单元格")
print("npu-smi  :", shutil.which("npu-smi") or "⚠️  未找到")
if shutil.which("npu-smi"):
    print()
    print(
        subprocess.run(["npu-smi", "info"], capture_output=True, text=True).stdout[:1800]
    )

## 8. 版本设计总览

四个版本的核函数全部写在同一个 `.asc` 文件中，共用同一份 Host 侧代码、同一组数据与同一套计时逻辑，因此版本之间的差异只体现在核函数本身，性能与精度数据都具有可比性。

<!-- markdown 版本（保留备用；如需切回，删除本注释标记并注释掉下方 HTML 表格）
| 版本 | 新增的唯一概念 | 算子类 | 核数 | 每核规约次数 | 核间合并方式 |
| --- | --- | --- | --- | --- | --- |
| **v1** | `ReduceSum` 与 `TBuf` 临时空间 | `KernelReduceTileWise` | 1 | 分块数（512 次） | 不涉及 |
| **v2** | 向量累加器 | `KernelReduceVecAccum` | 1 | **1 次** | 不涉及 |
| **v3** | workspace 两阶段合并 | `KernelReduceVecAccum` + `KernelReduceMerge` | 8 | 1 次 | 写 workspace + 二次启动 |
| **v4** | `SetAtomicAdd` 原子累加 | `KernelReduceVecAccum` | 8 | 1 次 | **原子累加到输出** |
-->

<table style="margin-left:0; margin-right:auto; border-collapse:collapse; text-align:left;">
<thead>
<tr>
<th style="border:1px solid #cccccc; padding:6px 12px; text-align:left; background-color:#f5f5f5;">版本</th>
<th style="border:1px solid #cccccc; padding:6px 12px; text-align:left; background-color:#f5f5f5;">新增的唯一概念</th>
<th style="border:1px solid #cccccc; padding:6px 12px; text-align:left; background-color:#f5f5f5;">算子类</th>
<th style="border:1px solid #cccccc; padding:6px 12px; text-align:left; background-color:#f5f5f5;">核数</th>
<th style="border:1px solid #cccccc; padding:6px 12px; text-align:left; background-color:#f5f5f5;">每核规约次数</th>
<th style="border:1px solid #cccccc; padding:6px 12px; text-align:left; background-color:#f5f5f5;">核间合并方式</th>
</tr>
</thead>
<tbody>
<tr>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;"><strong>v1</strong></td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;"><code>ReduceSum</code> 与 <code>TBuf</code> 临时空间</td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;"><code>KernelReduceTileWise</code></td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">1</td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">分块数（512 次）</td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">不涉及</td>
</tr>
<tr>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;"><strong>v2</strong></td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">向量累加器</td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;"><code>KernelReduceVecAccum</code></td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">1</td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;"><strong>1 次</strong></td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">不涉及</td>
</tr>
<tr>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;"><strong>v3</strong></td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">workspace 两阶段合并</td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;"><code>KernelReduceVecAccum</code> + <code>KernelReduceMerge</code></td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">8</td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">1 次</td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">写 workspace + 二次启动</td>
</tr>
<tr>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;"><strong>v4</strong></td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;"><code>SetAtomicAdd</code> 原子累加</td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;"><code>KernelReduceVecAccum</code></td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">8</td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">1 次</td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;"><strong>原子累加到输出</strong></td>
</tr>
</tbody>
</table>

**两条线索**：

- **v1 → v2**：改进**核内**规约，减少规约指令的执行次数。属于单核内部的优化。
- **v3 → v4**：改进**核间**合并，省去一次核函数启动与一趟 Global Memory 往返。

v2 → v3 这一步同时引入了多核，其加速主要来自并行度的提升，与前两个实验中 v1 → v2 的性质相同。

**三个算子类的对应关系**：v2、v3 的第一阶段与 v4 的核内计算逐字相同——都是「向量累加器 + 末尾一次规约」，三者的差别只在结果写到哪里、以及写的时候是否开启原子累加。因此它们共用 `KernelReduceVecAccum` 一个类，由 `Init` 的输出偏移与 `Process` 的原子累加开关区分。v1 与 v3 的第二阶段各有一个类。

下面的代码按 **Device 侧三个算子类 + 核函数入口** 与 **Host 侧基础设施 + 主程序** 的顺序写入同一个文件。

## 9. Device 侧实现

### 9.1 文件头与参数

先写入头文件与全部可调参数。`TOTAL_LENGTH` 只是元素总数的**默认值**，实际取值由命令行参数决定；`TILE_LENGTH` 与 `BLOCK_DIM` 是编译期常量，参数扫描时用 `bisheng -D` 覆盖。

In [ ]:
%%writefile src_reduce/ascendc_reduce_sum.asc
/**
 * 并行计算 第六章 实验三：规约算子 ReduceSum
 *
 * 本文件包含四个版本的核函数、CPU 基准、数据生成、精度校验与 main，
 * 由一条 bisheng 命令编译为单个可执行程序：
 *
 *   bisheng src_reduce/ascendc_reduce_sum.asc --npu-arch=dav-2201 -O2 -o src_reduce/ascendc_reduce_sum
 *
 * 用法：
 *   ./ascendc_reduce_sum                四个版本对照，N 取默认值 TOTAL_LENGTH
 *   ./ascendc_reduce_sum <N>            指定元素总数
 *   ./ascendc_reduce_sum <N> <seed>     指定元素总数与数据种子（精度分析用）
 */
#include <cstdio>
#include <cstdint>
#include <cstdlib>
#include <cmath>
#include <ctime>  // 计时：clock_gettime
#include <vector>

#include "acl/acl.h"          // Host 侧
#include "kernel_operator.h"  // Device 侧

/* ===================== 可由命令行 -D 覆盖的参数 ===================== */

using LabDType = float; /* 采用 float，与前两个实验的精度校验口径一致 */

/* 元素总数的默认值：2^21 = 2 097 152。实际取值由命令行参数决定 */
#ifndef LAB_TOTAL_LENGTH
#define LAB_TOTAL_LENGTH (2 * 1024 * 1024)
#endif
constexpr uint32_t TOTAL_LENGTH = static_cast<uint32_t>(LAB_TOTAL_LENGTH);

/* 分块长度：4096 个 float = 16 KB，同时也是向量累加器的宽度。
 * 官方的经验值是单次搬运 16 KB 以上通常能较好地发挥带宽，本取值恰在该门限上 */
#ifndef LAB_TILE_LENGTH
#define LAB_TILE_LENGTH (4 * 1024)
#endif
constexpr uint32_t TILE_LENGTH = static_cast<uint32_t>(LAB_TILE_LENGTH);

/* 参与计算的核数（v3/v4 使用；v1、v2 固定为 1） */
#ifndef LAB_BLOCK_DIM
#define LAB_BLOCK_DIM (8)
#endif
constexpr uint32_t BLOCK_DIM = static_cast<uint32_t>(LAB_BLOCK_DIM);

/* 计时参数：预热次数与重复次数 */
#ifndef LAB_REPEAT
#define LAB_REPEAT (50)
#endif
constexpr int32_t REPEAT = static_cast<int32_t>(LAB_REPEAT);
constexpr int32_t WARMUP = 3;

/* 对齐常量：搬运以 32 字节为单位，对 float 即 8 个元素 */
constexpr uint32_t ALIGN_ELEM = 32 / sizeof(LabDType);

/* 队列深度：TQue 模板的第二个参数，编译期常量 */
constexpr uint32_t QUEUE_DEPTH = 2;

/* 输出占用的元素数。规约结果只有 1 个标量，但搬运以 32 字节为基本单位，
 * 因此输出缓冲取 8 个元素，仅第 0 个有效 */
constexpr uint32_t OUT_LENGTH = ALIGN_ELEM;

/* workspace：每个核占 8 个元素（32 字节），第 0 个存部分和，其余置零 */
constexpr uint32_t WS_BLOCK = ALIGN_ELEM;
constexpr uint32_t WS_LENGTH = BLOCK_DIM * WS_BLOCK;
constexpr uint32_t WS_BYTES = WS_LENGTH * sizeof(LabDType);

/* ReduceSum 临时缓冲的元素数，按 2.1 节的核算式写成表达式：
 *   elementsPerRepeat = 256 / sizeof(float) = 64
 *   firstMaxRepeat    = TILE_LENGTH / 64
 *   tmpElements       = RoundUp(firstMaxRepeat, 8) * 8
 * TILE_LENGTH = 4096 时算得 512。写成表达式后，改 TILE_LENGTH 无须手工重算 */
constexpr uint32_t TMP_LENGTH = ((TILE_LENGTH / 64 + 7) / 8) * 8 * 8;

### 9.2 v1：逐块规约 `KernelReduceTileWise`

本版是单核基线，引入两个新构件：`ReduceSum` 接口，以及承载其临时空间的 `TBuf`。

流水结构仍是实验二的三段式，但 `Compute` 的内容变了：不再是逐元素相加后搬出，而是规约成一个标量、累加到成员变量上；只有全部分块处理完毕之后才执行一次 `CopyOut`。

注意 `CopyOut` 中的两行：规约结果只有 1 个有效值，但搬运以 32 字节为单位，因此先把 8 个元素整体置零，再写入第 0 个位置。

In [ ]:
%%writefile -a src_reduce/ascendc_reduce_sum.asc
/* ===================== v1：单核 · 逐块规约 =====================
 * 新增概念：ReduceSum 接口与 VECCALC 位置的 TBuf 临时缓冲
 *   每搬入一个分块即调用一次 ReduceSum 得到该块的和，再以标量形式累加。
 *   规约指令的执行次数等于分块数，串行累加链的长度也等于分块数。
 * 本版约束：仅使用一个核，作为后续版本加速比的基准。
 */
class KernelReduceTileWise {
 public:
  __aicore__ inline KernelReduceTileWise() {}

  __aicore__ inline void Init(GM_ADDR x, GM_ADDR z, uint32_t n) {
    tileNum_ = n / TILE_LENGTH;
    xGm.SetGlobalBuffer(reinterpret_cast<__gm__ LabDType *>(x), n);
    zGm.SetGlobalBuffer(reinterpret_cast<__gm__ LabDType *>(z), OUT_LENGTH);

    /* 参与流水传递的缓冲用 TQue：三个参数为队列对象、缓冲块数、单块字节数 */
    pipe.InitBuffer(inQueueX, QUEUE_DEPTH, TILE_LENGTH * sizeof(LabDType));
    pipe.InitBuffer(outQueueZ, 1, OUT_LENGTH * sizeof(LabDType));

    /* 只在 Compute 内部使用的缓冲用 TBuf：没有缓冲块数，也没有同步语义 */
    pipe.InitBuffer(tmpBuf, TMP_LENGTH * sizeof(LabDType));
    pipe.InitBuffer(dstBuf, ALIGN_ELEM * sizeof(LabDType));
  }

  __aicore__ inline void Process() {
    sum_ = static_cast<LabDType>(0);
    for (uint32_t i = 0; i < tileNum_; ++i) {
      CopyIn(i);
      Compute();
    }
    CopyOut(); /* 全部分块处理完毕后才搬出，整个核函数只搬出一次 */
  }

 private:
  __aicore__ inline void CopyIn(uint32_t progress) {
    AscendC::LocalTensor<LabDType> xLocal = inQueueX.AllocTensor<LabDType>();
    AscendC::DataCopy(xLocal, xGm[progress * TILE_LENGTH], TILE_LENGTH);
    inQueueX.EnQue(xLocal);
  }

  /* 每块规约一次，结果以标量累加。规约共执行 tileNum_ 次 */
  __aicore__ inline void Compute() {
    AscendC::LocalTensor<LabDType> xLocal = inQueueX.DeQue<LabDType>();
    /* TBuf 用 Get 取用，不需要 AllocTensor / FreeTensor */
    AscendC::LocalTensor<LabDType> tmpLocal = tmpBuf.Get<LabDType>();
    AscendC::LocalTensor<LabDType> dstLocal = dstBuf.Get<LabDType>();

    AscendC::ReduceSum(dstLocal, xLocal, tmpLocal, TILE_LENGTH);

    /* GetValue 属于 Scalar 流水操作，与上一条矢量指令之间存在数据依赖。
     * 算子工程使能自动同步时，无须手工插入 V_S 同步事件 */
    sum_ += dstLocal.GetValue(0);

    inQueueX.FreeTensor(xLocal);
  }

  /* 输出只有 1 个有效值，但搬运以 32 字节为单位：先整体置零，再写第 0 位 */
  __aicore__ inline void CopyOut() {
    AscendC::LocalTensor<LabDType> zLocal = outQueueZ.AllocTensor<LabDType>();
    AscendC::Duplicate(zLocal, static_cast<LabDType>(0), OUT_LENGTH);
    zLocal.SetValue(0, sum_);
    outQueueZ.EnQue(zLocal);

    AscendC::LocalTensor<LabDType> outLocal = outQueueZ.DeQue<LabDType>();
    AscendC::DataCopy(zGm, outLocal, OUT_LENGTH);
    outQueueZ.FreeTensor(outLocal);
  }

  AscendC::TPipe pipe;
  AscendC::TQue<AscendC::TPosition::VECIN, QUEUE_DEPTH> inQueueX;
  AscendC::TQue<AscendC::TPosition::VECOUT, 1> outQueueZ; /* 只搬出一次 */
  AscendC::TBuf<AscendC::TPosition::VECCALC> tmpBuf; /* ReduceSum 临时空间 */
  AscendC::TBuf<AscendC::TPosition::VECCALC> dstBuf; /* 单次规约的结果 */
  AscendC::GlobalTensor<LabDType> xGm, zGm;
  uint32_t tileNum_ = 0;
  LabDType sum_ = static_cast<LabDType>(0);
};

### 9.3 v2、v3 第一阶段与 v4 共用：向量累加器 `KernelReduceVecAccum`

这个类同时是 v2、v3 第一阶段和 v4 的实现。三者的核内计算逐字相同，差别只有两处，因此都做成参数：

<!-- markdown 版本（保留备用；如需切回，删除本注释标记并注释掉下方 HTML 表格）
| 参数 | v2 | v3 第一阶段 | v4 |
| --- | --- | --- | --- |
| 输出偏移 `outOffset` | 0（写到输出） | `blockIdx × WS_BLOCK`（写到 workspace） | 0（写到输出） |
| 原子累加开关 `atomicAdd` | `false` | `false` | **`true`** |
| 启动核数 | 1 | `BLOCK_DIM` | `BLOCK_DIM` |
-->

<table style="margin-left:0; margin-right:auto; border-collapse:collapse; text-align:left;">
<thead>
<tr>
<th style="border:1px solid #cccccc; padding:6px 12px; text-align:left; background-color:#f5f5f5;">参数</th>
<th style="border:1px solid #cccccc; padding:6px 12px; text-align:left; background-color:#f5f5f5;">v2</th>
<th style="border:1px solid #cccccc; padding:6px 12px; text-align:left; background-color:#f5f5f5;">v3 第一阶段</th>
<th style="border:1px solid #cccccc; padding:6px 12px; text-align:left; background-color:#f5f5f5;">v4</th>
</tr>
</thead>
<tbody>
<tr>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">输出偏移 <code>outOffset</code></td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">0（写到输出）</td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;"><code>blockIdx × WS_BLOCK</code>（写到 workspace）</td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">0（写到输出）</td>
</tr>
<tr>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">原子累加开关 <code>atomicAdd</code></td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;"><code>false</code></td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;"><code>false</code></td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">**<code>true</code>**</td>
</tr>
<tr>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">启动核数</td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">1</td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;"><code>BLOCK_DIM</code></td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;"><code>BLOCK_DIM</code></td>
</tr>
</tbody>
</table>

核间切分沿用实验二的做法：由 `GetBlockIdx()` 算出偏移，写进 `SetGlobalBuffer`，核内其余代码保持不变。当核数为 1 时 `GetBlockNum()` 返回 1，偏移为 0，因此同一份代码也覆盖了 v2 的单核情形。

`Process` 的三步结构值得单独记住：**清零累加器 → 循环内只做 `Add` → 循环外只做一次 `ReduceSum`**。

**片上缓冲占用的核算**（沿用实验二 §3.1 的做法，编写代码之前先算）：本类共申请五块缓冲，`TILE_LENGTH = 4096` 时为 输入队列 4096 × 4 B × 2 块 = 32 KB、累加器 4096 × 4 B = 16 KB、`ReduceSum` 临时空间 512 × 4 B = 2 KB，另加两块各 32 B 的输出与规约结果，合计约 50 KB，占 UB（192 KB）的 26%。与实验二不同的是，这里多出了累加器和临时空间两项，它们都随 `TILE_LENGTH` 线性增长，因此动手练习第 5 题增大分块长度时需要重新核算。

In [ ]:
%%writefile -a src_reduce/ascendc_reduce_sum.asc
/* ============ v2 / v3 第一阶段 / v4：向量累加器 ============
 * 新增概念：部分和保留在向量中，末尾只规约一次
 *   循环内仅执行 Add(acc, acc, x, TILE_LENGTH)，把 TILE_LENGTH 条通道
 *   同时向前推进；ReduceSum 移出循环，全程只执行一次。
 * 对照：与第三章 GEMV 实验中「水平规约仅在每行末尾执行一次」同构，
 *       区别仅在于通道数由 4 条扩大到 TILE_LENGTH 条。
 */
class KernelReduceVecAccum {
 public:
  __aicore__ inline KernelReduceVecAccum() {}

  /* outOffset：结果写到目标张量的第几个元素
   *   v2 / v4   传 0                      —— 直接写到输出
   *   v3 阶段一 传 blockIdx * WS_BLOCK    —— 写到 workspace 中属于本核的 32 字节 */
  __aicore__ inline void Init(GM_ADDR x, GM_ADDR out, uint32_t n,
                              uint32_t outOffset) {
    /* 核间切分：各核处理长度相同、互不重叠的一段 */
    blockLength_ = n / AscendC::GetBlockNum();
    tileNum_ = blockLength_ / TILE_LENGTH;
    const uint32_t offset = AscendC::GetBlockIdx() * blockLength_;

    xGm.SetGlobalBuffer(reinterpret_cast<__gm__ LabDType *>(x) + offset,
                        blockLength_);
    outGm.SetGlobalBuffer(reinterpret_cast<__gm__ LabDType *>(out) + outOffset,
                          ALIGN_ELEM);

    pipe.InitBuffer(inQueueX, QUEUE_DEPTH, TILE_LENGTH * sizeof(LabDType));
    pipe.InitBuffer(outQueue, 1, ALIGN_ELEM * sizeof(LabDType));
    pipe.InitBuffer(accBuf, TILE_LENGTH * sizeof(LabDType)); /* 向量累加器 */
    pipe.InitBuffer(tmpBuf, TMP_LENGTH * sizeof(LabDType));
    pipe.InitBuffer(dstBuf, ALIGN_ELEM * sizeof(LabDType));
  }

  /* atomicAdd 为 true 时搬出改为原子累加（v4），为 false 时为普通覆盖写 */
  __aicore__ inline void Process(bool atomicAdd) {
    AscendC::LocalTensor<LabDType> acc = accBuf.Get<LabDType>();

    /* 累加器清零：必须在循环之前，且只能做一次。
     * 若误置于循环内部，每轮都会清掉此前累积的部分和 */
    AscendC::Duplicate(acc, static_cast<LabDType>(0), TILE_LENGTH);

    for (uint32_t i = 0; i < tileNum_; ++i) {
      CopyIn(i);
      Accumulate(acc);
    }

    Finalize(acc, atomicAdd);
  }

 private:
  __aicore__ inline void CopyIn(uint32_t progress) {
    AscendC::LocalTensor<LabDType> xLocal = inQueueX.AllocTensor<LabDType>();
    AscendC::DataCopy(xLocal, xGm[progress * TILE_LENGTH], TILE_LENGTH);
    inQueueX.EnQue(xLocal);
  }

  /* 循环内：只做逐元素累加，不做规约 */
  __aicore__ inline void Accumulate(const AscendC::LocalTensor<LabDType> &acc) {
    AscendC::LocalTensor<LabDType> xLocal = inQueueX.DeQue<LabDType>();
    AscendC::Add(acc, acc, xLocal, TILE_LENGTH);
    inQueueX.FreeTensor(xLocal);
  }

  /* 循环外：唯一一次规约，随后写出结果 */
  __aicore__ inline void Finalize(const AscendC::LocalTensor<LabDType> &acc,
                                  bool atomicAdd) {
    AscendC::LocalTensor<LabDType> tmpLocal = tmpBuf.Get<LabDType>();
    AscendC::LocalTensor<LabDType> dstLocal = dstBuf.Get<LabDType>();
    AscendC::ReduceSum(dstLocal, acc, tmpLocal, TILE_LENGTH);

    AscendC::LocalTensor<LabDType> outLocal = outQueue.AllocTensor<LabDType>();
    AscendC::Duplicate(outLocal, static_cast<LabDType>(0), ALIGN_ELEM);
    outLocal.SetValue(0, dstLocal.GetValue(0));
    outQueue.EnQue(outLocal);

    AscendC::LocalTensor<LabDType> o = outQueue.DeQue<LabDType>();
    if (atomicAdd) {
      /* 开启后，本核的搬出由覆盖写改为在已有值上累加 */
      AscendC::SetAtomicAdd<LabDType>();
    }
    AscendC::DataCopy(outGm, o, ALIGN_ELEM);
    if (atomicAdd) {
      /* 用完立即关闭，否则会影响此后所有的搬出指令 */
      AscendC::SetAtomicNone();
    }
    outQueue.FreeTensor(o);
  }

  AscendC::TPipe pipe;
  AscendC::TQue<AscendC::TPosition::VECIN, QUEUE_DEPTH> inQueueX;
  AscendC::TQue<AscendC::TPosition::VECOUT, 1> outQueue;
  AscendC::TBuf<AscendC::TPosition::VECCALC> accBuf;
  AscendC::TBuf<AscendC::TPosition::VECCALC> tmpBuf;
  AscendC::TBuf<AscendC::TPosition::VECCALC> dstBuf;
  AscendC::GlobalTensor<LabDType> xGm, outGm;
  uint32_t blockLength_ = 0;
  uint32_t tileNum_ = 0;
};

### 9.4 v3 第二阶段：合并 workspace `KernelReduceMerge`

第二阶段由单个核完成，把 workspace 中的 `BLOCK_DIM × 8` 个元素整块搬入后规约一次。填充位置本身是零，因此不需要区分有效值与填充值——这正是 §4.1 中那个布局的用意。

这个类的工作量极小（8 核时只有 64 个元素），它的耗时几乎全部来自一次核函数启动和一趟 Global Memory 往返。v4 要省掉的就是这两项。

In [ ]:
%%writefile -a src_reduce/ascendc_reduce_sum.asc
/* ============ v3 第二阶段：单核合并 workspace ============
 * 填充位置为零，直接整块规约即可，无须区分有效值与填充值。
 * 本阶段必须在第一阶段的全部核完成写入之后才能开始：同一 Stream 内的
 * 任务按下发顺序串行执行，因此两次启动就已保证顺序，不需要额外同步。
 */
class KernelReduceMerge {
 public:
  __aicore__ inline KernelReduceMerge() {}

  __aicore__ inline void Init(GM_ADDR ws, GM_ADDR z) {
    wsGm.SetGlobalBuffer(reinterpret_cast<__gm__ LabDType *>(ws), WS_LENGTH);
    zGm.SetGlobalBuffer(reinterpret_cast<__gm__ LabDType *>(z), OUT_LENGTH);

    pipe.InitBuffer(inQueue, 1, WS_LENGTH * sizeof(LabDType));
    pipe.InitBuffer(outQueue, 1, OUT_LENGTH * sizeof(LabDType));
    pipe.InitBuffer(tmpBuf, TMP_LENGTH * sizeof(LabDType));
    pipe.InitBuffer(dstBuf, ALIGN_ELEM * sizeof(LabDType));
  }

  __aicore__ inline void Process() {
    AscendC::LocalTensor<LabDType> wsLocal = inQueue.AllocTensor<LabDType>();
    AscendC::DataCopy(wsLocal, wsGm, WS_LENGTH);
    inQueue.EnQue(wsLocal);

    AscendC::LocalTensor<LabDType> in = inQueue.DeQue<LabDType>();
    AscendC::LocalTensor<LabDType> tmpLocal = tmpBuf.Get<LabDType>();
    AscendC::LocalTensor<LabDType> dstLocal = dstBuf.Get<LabDType>();
    AscendC::ReduceSum(dstLocal, in, tmpLocal, WS_LENGTH);
    inQueue.FreeTensor(in);

    AscendC::LocalTensor<LabDType> zLocal = outQueue.AllocTensor<LabDType>();
    AscendC::Duplicate(zLocal, static_cast<LabDType>(0), OUT_LENGTH);
    zLocal.SetValue(0, dstLocal.GetValue(0));
    outQueue.EnQue(zLocal);

    AscendC::LocalTensor<LabDType> o = outQueue.DeQue<LabDType>();
    AscendC::DataCopy(zGm, o, OUT_LENGTH);
    outQueue.FreeTensor(o);
  }

 private:
  AscendC::TPipe pipe;
  AscendC::TQue<AscendC::TPosition::VECIN, 1> inQueue;
  AscendC::TQue<AscendC::TPosition::VECOUT, 1> outQueue;
  AscendC::TBuf<AscendC::TPosition::VECCALC> tmpBuf;
  AscendC::TBuf<AscendC::TPosition::VECCALC> dstBuf;
  AscendC::GlobalTensor<LabDType> wsGm, zGm;
};

### 9.5 五个核函数入口

四个版本共需要五个入口，因为 v3 分两个阶段。每个入口的第一行都是 `KERNEL_TASK_TYPE_DEFAULT(KERNEL_TYPE_AIV_ONLY);`，含义是本核函数只使用矢量核（AIV）。在 AIC 与 AIV 分离的架构上，核函数的任务类型决定了运行时按哪一种口径分配计算单元：纯矢量类型按 AIV 个数分配，混合类型则按「组合」分配，一个组合包含 1 个 AIC 与 2 个 AIV。同一编译单元中存在多个核函数时任务类型不会自动推导，若被按混合口径处理，`GetBlockNum()` 返回的数值就不再等于启动时指定的 `blockDim`，核间切分与核数扫描的横轴含义都会随之出错。这一点与实验二 §7.5 的说明一致。

`reduce_v3_stage1` 与 `reduce_v4` 的区别只有两处：前者把结果写到 workspace 中属于本核的位置，后者把结果原子累加到同一段输出上。

In [ ]:
%%writefile -a src_reduce/ascendc_reduce_sum.asc
/* ===================== 核函数入口 ===================== */

extern "C" __global__ __aicore__ void reduce_v1(GM_ADDR x, GM_ADDR z,
                                                uint32_t n) {
  KERNEL_TASK_TYPE_DEFAULT(KERNEL_TYPE_AIV_ONLY); /* 声明为纯矢量内核 */
  KernelReduceTileWise op;
  op.Init(x, z, n);
  op.Process();
}

extern "C" __global__ __aicore__ void reduce_v2(GM_ADDR x, GM_ADDR z,
                                                uint32_t n) {
  KERNEL_TASK_TYPE_DEFAULT(KERNEL_TYPE_AIV_ONLY);
  KernelReduceVecAccum op;
  op.Init(x, z, n, 0); /* 单核启动，结果直接写到输出 */
  op.Process(false);
}

extern "C" __global__ __aicore__ void reduce_v3_stage1(GM_ADDR x, GM_ADDR ws,
                                                       uint32_t n) {
  KERNEL_TASK_TYPE_DEFAULT(KERNEL_TYPE_AIV_ONLY);
  KernelReduceVecAccum op;
  /* 部分和写到 workspace 中属于本核的 32 字节区间 */
  op.Init(x, ws, n, AscendC::GetBlockIdx() * WS_BLOCK);
  op.Process(false);
}

extern "C" __global__ __aicore__ void reduce_v3_stage2(GM_ADDR ws, GM_ADDR z) {
  KERNEL_TASK_TYPE_DEFAULT(KERNEL_TYPE_AIV_ONLY);
  KernelReduceMerge op;
  op.Init(ws, z);
  op.Process();
}

extern "C" __global__ __aicore__ void reduce_v4(GM_ADDR x, GM_ADDR z,
                                                uint32_t n) {
  KERNEL_TASK_TYPE_DEFAULT(KERNEL_TYPE_AIV_ONLY);
  KernelReduceVecAccum op;
  /* 全部核指向同一段输出，靠原子累加保证正确性 */
  op.Init(x, z, n, 0);
  op.Process(true);
}

## 10. Host 侧实现

Device 侧到此结束。Host 侧的代码分两部分写入：先是一组基础设施，然后是主程序。

### 10.1 基础设施：错误检查、计时、数据生成、参考真值、CPU 基准与结果校验

<!-- markdown 版本（保留备用；如需切回，删除本注释标记并注释掉下方 HTML 表格）
| 内容 | 说明 |
| --- | --- |
| `ACL_CHECK` | 与前两个实验相同，任何 ACL 接口返回非 0 即报告位置并退出 |
| 计时与设备管理 | `GetTimeMs` 用 `clock_gettime(CLOCK_MONOTONIC)` 取毫秒时刻；`NpuInit` / `NpuFinalize` 中的设备初始化开销在百毫秒量级，因此放在测量循环之外 |
| `GenerateInput` | 用固定种子的线性同余发生器生成 [0, 1) 上的数据，不读写任何文件，也不依赖 `<random>` |
| `GoldenSum` | 参考真值，`float64` 顺序累加 |
| `RunOnCpu` | CPU 单线程 `float32` 顺序累加，既是性能基准，也是精度对照中链最长的一端 |
| `VerifyScalar` | 用**相对误差**校验单个标量，同时打印该版本的链长与通道数 |
| `TIME_KERNEL` | 对应 §6 的 `kernel_ms`：先预热，再重复下发并逐次同步后取平均 |
| `RUN_VERSION` | 把「清零 → 运行一次并校验 → 再计时」的固定流程收成一个宏 |
-->

<table style="margin-left:0; margin-right:auto; border-collapse:collapse; text-align:left;">
<thead>
<tr>
<th style="border:1px solid #cccccc; padding:6px 12px; text-align:left; background-color:#f5f5f5;">内容</th>
<th style="border:1px solid #cccccc; padding:6px 12px; text-align:left; background-color:#f5f5f5;">说明</th>
</tr>
</thead>
<tbody>
<tr>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;"><code>ACL_CHECK</code></td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">与前两个实验相同，任何 ACL 接口返回非 0 即报告位置并退出</td>
</tr>
<tr>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">计时与设备管理</td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;"><code>GetTimeMs</code> 用 <code>clock_gettime(CLOCK_MONOTONIC)</code> 取毫秒时刻；<code>NpuInit</code> / <code>NpuFinalize</code> 中的设备初始化开销在百毫秒量级，因此放在测量循环之外</td>
</tr>
<tr>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;"><code>GenerateInput</code></td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">用固定种子的线性同余发生器生成 [0, 1) 上的数据，不读写任何文件，也不依赖 <code>&lt;random&gt;</code></td>
</tr>
<tr>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;"><code>GoldenSum</code></td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">参考真值，<code>float64</code> 顺序累加</td>
</tr>
<tr>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;"><code>RunOnCpu</code></td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">CPU 单线程 <code>float32</code> 顺序累加，既是性能基准，也是精度对照中链最长的一端</td>
</tr>
<tr>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;"><code>VerifyScalar</code></td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">用<strong>相对误差</strong>校验单个标量，同时打印该版本的链长与通道数</td>
</tr>
<tr>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;"><code>TIME_KERNEL</code></td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">对应 §6 的 <code>kernel_ms</code>：先预热，再重复下发并逐次同步后取平均</td>
</tr>
<tr>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;"><code>RUN_VERSION</code></td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">把「清零 → 运行一次并校验 → 再计时」的固定流程收成一个宏</td>
</tr>
</tbody>
</table>

In [ ]:
%%writefile -a src_reduce/ascendc_reduce_sum.asc
/* ============================================================
 *                       Host 侧代码
 * ============================================================ */

#define ACL_CHECK(expr)                                                       \
  do {                                                                        \
    aclError _ret = (expr);                                                   \
    if (_ret != ACL_SUCCESS) {                                                \
      std::printf("[ACL ERROR] %s:%d  %s  returned %d\n", __FILE__, __LINE__, \
                  #expr, static_cast<int>(_ret));                             \
      std::exit(EXIT_FAILURE);                                                \
    }                                                                         \
  } while (0)

/* 取当前时刻，单位毫秒。CLOCK_MONOTONIC 自系统启动起单调递增，
 * 不受 NTP 校时或手动改时间影响，是测量时间间隔的标准做法 */
static inline double GetTimeMs() {
  struct timespec ts;
  clock_gettime(CLOCK_MONOTONIC, &ts);
  return static_cast<double>(ts.tv_sec) * 1000.0 +
         static_cast<double>(ts.tv_nsec) / 1000000.0;
}

static int32_t g_deviceId = 0;
static aclrtStream g_stream = nullptr;

static void NpuInit() {
  ACL_CHECK(aclInit(nullptr));
  ACL_CHECK(aclrtSetDevice(g_deviceId));
  ACL_CHECK(aclrtCreateStream(&g_stream));
}

static void NpuFinalize() {
  ACL_CHECK(aclrtDestroyStream(g_stream));
  ACL_CHECK(aclrtResetDevice(g_deviceId));
  ACL_CHECK(aclFinalize());
}

/* ---------- 数据生成：线性同余发生器，输出 [0, 1) 上的均匀分布 ---------- */
static inline float LcgNextFloat(uint32_t &state) {
  state = state * 1664525u + 1013904223u; /* Numerical Recipes 推荐参数 */
  /* 取高 24 位（低位周期短），线性映射到 [0, 1) */
  return static_cast<float>(state >> 8) * (1.0f / 16777216.0f);
}

static void GenerateInput(std::vector<LabDType> &x, uint32_t seed) {
  uint32_t s = seed;
  for (size_t i = 0; i < x.size(); ++i) {
    x[i] = static_cast<LabDType>(LcgNextFloat(s));
  }
}

/* ---------- 参考真值：float64 顺序累加。参考值的精度必须高于被测对象 ---------- */
static double GoldenSum(const std::vector<LabDType> &x) {
  double s = 0.0;
  for (size_t i = 0; i < x.size(); ++i) {
    s += static_cast<double>(x[i]);
  }
  return s;
}

/* ---------- CPU 基准：单线程 float32 顺序累加 ----------
 * 累加器的初值取自 volatile 变量：编译器无法证明各轮初值相同，
 * 因而不能把整个重复循环提到外面。该变量恒为 0，不影响结果。 */
static volatile float g_accSeed = 0.0f;

static double RunOnCpu(const std::vector<LabDType> &x, float &result,
                       int warmup, int repeat) {
  const size_t n = x.size();

  for (int r = 0; r < warmup; ++r) {
    float s = g_accSeed;
    for (size_t i = 0; i < n; ++i) s += x[i];
    result = s;
  }

  const double t0 = GetTimeMs();
  for (int r = 0; r < repeat; ++r) {
    float s = g_accSeed;
    for (size_t i = 0; i < n; ++i) s += x[i];
    result = s;
  }
  return (GetTimeMs() - t0) / repeat;
}

/* ---------- 相对误差校验 ----------
 * 规约的输出只有一个标量，无法像逐元素算子那样统计超差元素数，
 * 因此改为直接比较相对误差。chain 与 lanes 是 5 节定义的两个量。 */
static bool VerifyScalar(const char *ver, uint32_t n, double got, double golden,
                         uint32_t chain, uint32_t lanes, double eps = 1e-4) {
  const double rel = std::fabs(got - golden) / std::fabs(golden);
  const bool ok = (rel <= eps);
  std::printf(
      "[VERIFY] ver=%s n=%u sum=%.6f golden=%.6f rel_err=%.3e chain=%u "
      "lanes=%u result=%s\n",
      ver, n, got, golden, rel, chain, lanes, ok ? "PASS" : "FAIL");
  return ok;
}

/* ---------- 打印一行可被程序解析的性能记录 ----------
 * 加速比与其分子、分母一并输出：脱离基准耗时的加速比无法解读，见 15 节第 5 条。 */
static void ReportPerf(const char *ver, uint32_t n, uint32_t blockDim,
                       uint32_t reduceCalls, double cpuMs, double kernelMs) {
  std::printf(
      "[PERF]   ver=%s n=%u blockDim=%u reduce_calls=%u cpu_ms=%.4f "
      "kernel_ms=%.4f sp_kernel=%.4f\n",
      ver, n, blockDim, reduceCalls, cpuMs, kernelMs, cpuMs / kernelMs);
}

/* ---------- 计时宏：核函数耗时（不含主机与设备之间的数据搬运） ---------- */
#define TIME_KERNEL(LAUNCH, OUT_MS)                                               \
  do {                                                                            \
    for (int _w = 0; _w < WARMUP; ++_w) {                                         \
      LAUNCH;                                                                     \
      ACL_CHECK(aclrtSynchronizeStream(g_stream));                                \
    }                                                                             \
    const double _t0 = GetTimeMs();                                               \
    for (int _r = 0; _r < REPEAT; ++_r) {                                         \
      LAUNCH;                                                                     \
      ACL_CHECK(aclrtSynchronizeStream(g_stream)); /* 先同步再停止计时 */ \
    }                                                                             \
    (OUT_MS) = (GetTimeMs() - _t0) / REPEAT;                                      \
  } while (0)

### 10.2 主程序

主程序的结构与前两个实验一致：解析命令行、生成数据、算参考值与 CPU 基准、申请显存、逐版本运行、释放资源。

有两处是本实验特有的：

**长度约束的检查放在最前面。** 本实验要求元素总数能被 `BLOCK_DIM × TILE_LENGTH` 整除。若不满足，`tileNum` 会被整数除法截断——极端情况下变成 0，核函数什么都不做却正常返回，得到的是一个结果严重错误、却以成功状态退出的程序。因此在 `main` 的开头就直接检查并报错退出：**让错误在最早的位置以最明显的方式暴露。**

**四个版本共用 `RUN_VERSION` 宏。** 它把「清零输出 → 运行一次并校验 → 计时」固定下来，同时把该版本的链长与通道数一并传入，使 `[VERIFY]` 行自带 §5 所需的两个量。

In [ ]:
%%writefile -a src_reduce/ascendc_reduce_sum.asc
/* 每个版本的固定流程：清零输出 → 运行一次并校验 → 再计时。
 * 取结果必须在计时之前：v4 的原子累加会把计时循环中的 REPEAT 次结果全部叠加。 */
#define RUN_VERSION(TAG, LAUNCH, BD, CALLS, CHAIN, LANES)                    \
  do {                                                                       \
    ACL_CHECK(aclrtMemset(zDev, outBytes, 0, outBytes));                     \
    LAUNCH;                                                                  \
    ACL_CHECK(aclrtSynchronizeStream(g_stream));                             \
    ACL_CHECK(aclrtMemcpy(zHost, outBytes, zDev, outBytes,                   \
                          ACL_MEMCPY_DEVICE_TO_HOST));                       \
    allPass &= VerifyScalar(                                                 \
        TAG, n, static_cast<double>(reinterpret_cast<LabDType *>(zHost)[0]), \
        golden, CHAIN, LANES);                                               \
    TIME_KERNEL(LAUNCH, kMs);                                                \
    ReportPerf(TAG, n, BD, CALLS, cpuMs, kMs);                               \
  } while (0)

int32_t main(int argc, char *argv[]) {
  /* ---------- 解析命令行：元素总数与数据种子都是运行时参数 ---------- */
  uint32_t n = TOTAL_LENGTH;
  uint32_t seed = 2026u;
  if (argc > 1) n = static_cast<uint32_t>(std::strtoul(argv[1], nullptr, 10));
  if (argc > 2)
    seed = static_cast<uint32_t>(std::strtoul(argv[2], nullptr, 10));

  /* 长度约束：核间切分与核内分块都必须整除，否则 tileNum 会被截断。
   * 让错误在最早的位置暴露，而不是算出一个错误结果却以成功状态退出 */
  const uint32_t granularity = BLOCK_DIM * TILE_LENGTH;
  if (n == 0 || n % granularity != 0) {
    std::printf(
        "[FATAL] N 必须是 BLOCK_DIM x TILE_LENGTH = %u 的整数倍，当前 N = %u\n",
        granularity, n);
    return 1;
  }

  /* ---------- 生成数据、算参考真值与 CPU 基准 ---------- */
  std::vector<LabDType> x(n);
  GenerateInput(x, seed);

  const double golden = GoldenSum(x);
  float cpuSum = 0.0f;
  const int cpuRepeat = (REPEAT <= 1) ? 1 : ((n > (4u << 20)) ? 5 : 20);
  const double cpuMs =
      RunOnCpu(x, cpuSum, (REPEAT <= 1) ? 1 : WARMUP, cpuRepeat);
  const double cpuRel =
      std::fabs(static_cast<double>(cpuSum) - golden) / std::fabs(golden);

  std::printf("N=%u  TILE_LENGTH=%u  BLOCK_DIM=%u  REPEAT=%d  seed=%u\n", n,
              TILE_LENGTH, BLOCK_DIM, REPEAT, seed);
  std::printf(
      "[BASE]   n=%u golden_f64=%.6f cpu_f32=%.6f cpu_rel_err=%.3e chain=%u "
      "lanes=1 cpu_ms=%.4f\n",
      n, golden, static_cast<double>(cpuSum), cpuRel, n, cpuMs);

  /* ---------- 申请显存并搬入输入 ---------- */
  NpuInit();
  const size_t bytes = static_cast<size_t>(n) * sizeof(LabDType);
  const size_t outBytes = OUT_LENGTH * sizeof(LabDType);
  void *hostX = x.data();
  uint8_t *xDev = nullptr, *zDev = nullptr, *wsDev = nullptr, *zHost = nullptr;
  ACL_CHECK(aclrtMalloc((void **)&xDev, bytes, ACL_MEM_MALLOC_HUGE_FIRST));
  ACL_CHECK(aclrtMalloc((void **)&zDev, outBytes, ACL_MEM_MALLOC_HUGE_FIRST));
  ACL_CHECK(aclrtMalloc((void **)&wsDev, WS_BYTES, ACL_MEM_MALLOC_HUGE_FIRST));
  ACL_CHECK(aclrtMallocHost((void **)&zHost, outBytes));
  ACL_CHECK(aclrtMemcpy(xDev, bytes, hostX, bytes, ACL_MEMCPY_HOST_TO_DEVICE));

  bool allPass = true;
  double kMs = 0.0;

  /* chain 与 lanes 见 5 节：单条通道的串行链长，以及独立通道的条数 */
  const uint32_t tilesTotal = n / TILE_LENGTH;
  const uint32_t tilesPerCore = n / granularity;

  /* ---------------- v1：单核 · 逐块规约 ---------------- */
  RUN_VERSION("v1", (reduce_v1<<<1, nullptr, g_stream>>>(xDev, zDev, n)), 1,
              tilesTotal, tilesTotal, 1);

  /* ---------------- v2：单核 · 向量累加器 ---------------- */
  RUN_VERSION("v2", (reduce_v2<<<1, nullptr, g_stream>>>(xDev, zDev, n)), 1, 1,
              tilesTotal, TILE_LENGTH);

  /* ---------------- v3：多核 + workspace 两阶段 ---------------- */
  RUN_VERSION(
      "v3",
      (reduce_v3_stage1<<<BLOCK_DIM, nullptr, g_stream>>>(xDev, wsDev, n),
       reduce_v3_stage2<<<1, nullptr, g_stream>>>(wsDev, zDev)),
      BLOCK_DIM, 1, tilesPerCore, granularity);

  /* ---------------- v4：多核 + 原子累加 ---------------- */
  RUN_VERSION("v4",
              (reduce_v4<<<BLOCK_DIM, nullptr, g_stream>>>(xDev, zDev, n)),
              BLOCK_DIM, 1, tilesPerCore, granularity);

  /* ---------- 释放资源（与申请严格成对，顺序相反） ---------- */
  ACL_CHECK(aclrtFreeHost(zHost));
  ACL_CHECK(aclrtFree(wsDev));
  ACL_CHECK(aclrtFree(zDev));
  ACL_CHECK(aclrtFree(xDev));
  NpuFinalize();

  std::printf(allPass ? "[SUCCESS] 全部版本校验通过。\n"
                      : "[FAILED] 存在校验未通过的版本！\n");
  return allPass ? 0 : 1;
}

## 11. 编译与运行

`-O2` 不能省略——CPU 基准与 NPU 代码在同一次编译中生成，使用 `-O0` 会人为放大基准实现的耗时，使加速比失真。

In [ ]:
import subprocess

ARCH = "dav-2201"  # ← 若设备不是 Atlas A2/A3，请按实验一 §7.2 的表修改
SRC = "src_reduce/ascendc_reduce_sum.asc"
EXE = "src_reduce/ascendc_reduce_sum"

# bisheng [算子源文件] --npu-arch=[NPU架构版本号] -O2 -o [输出产物名称]
cmd = ["bisheng", SRC, "--npu-arch=" + ARCH, "-O2", "-o", EXE]
print("$ " + " ".join(cmd))

proc = subprocess.run(cmd, capture_output=True, text=True)
msg = (proc.stdout + proc.stderr).strip()
if msg:
    print(msg)

print(
    "✅ 编译成功"
    if proc.returncode == 0
    else "❌ 编译失败（返回码 %d）" % proc.returncode
)

一次运行即可输出四个版本的校验结果与性能记录。

In [ ]:
def run_demo(args=(), exe=None, timeout=900):
    # 与实验二同名同约定：只返回 stdout；exe 缺省为 §11 编译出的主程序
    proc = subprocess.run(
        ["./" + (exe or EXE)] + [str(a) for a in args],
        capture_output=True,
        text=True,
        timeout=timeout,
    )
    if proc.returncode != 0 and not proc.stdout:
        print("返回码", proc.returncode)
        print(proc.stderr)
    return proc.stdout


out_main = run_demo()
print(out_main)

### 11.1 解析输出

三类记录行的字段名固定，用一行正则即可提取为字典。

In [ ]:
def parse_rows(text, tag):
    # 把所有以 [tag] 开头的记录行解析为 dict 列表
    rows = []
    for line in text.splitlines():
        if line.startswith("[" + tag + "]"):
            d = {}
            for kv in line.split()[1:]:
                k, v = kv.split("=", 1)
                d[k] = v if k in ("ver", "result") else float(v)
            rows.append(d)
    return rows


def parse_perf(text):
    return parse_rows(text, "PERF")


def parse_verify(text):
    return {r["ver"]: r for r in parse_rows(text, "VERIFY")}


def parse_base(text):
    rows = parse_rows(text, "BASE")
    return rows[0] if rows else None


rows_main = parse_perf(out_main)
chk_main = parse_verify(out_main)
base_main = parse_base(out_main)
base_v1 = rows_main[0]["kernel_ms"] if rows_main else 1.0

print("参考真值 (float64) = %.6f" % base_main["golden_f64"])
print(
    "CPU float32 顺序累加 = %.6f   相对误差 = %.3e   链长 = %d   通道数 = 1"
    % (base_main["cpu_f32"], base_main["cpu_rel_err"], base_main["chain"])
)
print()
print(
    "%-4s %5s %8s %10s %11s %9s %8s %11s %7s %8s %6s"
    % (
        "版本",
        "核数",
        "规约次数",
        "CPU(ms)",
        "kernel(ms)",
        "vs CPU",
        "vs v1",
        "相对误差",
        "链长",
        "通道数",
        "校验",
    )
)
print("-" * 108)
for r in rows_main:
    v = chk_main[r["ver"]]
    print(
        "%-4s %5d %8d %10.4f %11.4f %8.2fx %7.2fx %11.3e %7d %8d %6s"
        % (
            r["ver"],
            r["blockDim"],
            r["reduce_calls"],
            r["cpu_ms"],
            r["kernel_ms"],
            r["sp_kernel"],
            base_v1 / r["kernel_ms"],
            v["rel_err"],
            v["chain"],
            v["lanes"],
            v["result"],
        )
    )

## 12. 结果可视化

两张加速比图回答两个不同的问题：

<!-- markdown 版本（保留备用；如需切回，删除本注释标记并注释掉下方 HTML 表格）
| 图 | 基线 | 回答的问题 |
| --- | --- | --- |
| 图一 | **CPU 单线程 float32 顺序累加** | 使用 NPU 的整体收益 |
| 图二 | **v1（单核逐块规约）** | 每一项优化各自的贡献 |
-->

<table style="margin-left:0; margin-right:auto; border-collapse:collapse; text-align:left;">
<thead>
<tr>
<th style="border:1px solid #cccccc; padding:6px 12px; text-align:left; background-color:#f5f5f5;">图</th>
<th style="border:1px solid #cccccc; padding:6px 12px; text-align:left; background-color:#f5f5f5;">基线</th>
<th style="border:1px solid #cccccc; padding:6px 12px; text-align:left; background-color:#f5f5f5;">回答的问题</th>
</tr>
</thead>
<tbody>
<tr>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">图一</td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;"><strong>CPU 单线程 float32 顺序累加</strong></td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">使用 NPU 的整体收益</td>
</tr>
<tr>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">图二</td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;"><strong>v1（单核逐块规约）</strong></td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">每一项优化各自的贡献</td>
</tr>
</tbody>
</table>

图一以核函数耗时为口径，并画一条 y = 1 的基准线作为参照。读这张图时须同时对照结果表中的 `cpu_ms` 一列：加速比的高低既取决于 NPU 侧，也取决于基准实现的快慢，§15 ⑤ 会对此作出说明。

In [ ]:
%matplotlib inline
import matplotlib
import matplotlib.pyplot as plt
import numpy as np

matplotlib.rcParams["font.sans-serif"] = ["DejaVu Sans"]  # 图内标签统一使用 ASCII
matplotlib.rcParams["axes.unicode_minus"] = False

C_KERNEL, C_ALT, C_BASE, C_OPT = "#3B6FE0", "#E07A3B", "#9AA5B1", "#2E9E6B"

vers = [r["ver"] for r in rows_main]
spk = [r["sp_kernel"] for r in rows_main]
xpos = np.arange(len(vers))

fig, ax = plt.subplots(figsize=(7.6, 4.3), dpi=120)
ax.bar(xpos, spk, 0.5, color=C_KERNEL, label="NPU kernel only")
ax.axhline(1.0, color="#888888", lw=1.0, ls=":")
ax.text(-0.45, 1.06, "baseline: CPU 1 thread = 1.0x", fontsize=9, color="#777777")
for i, a in enumerate(spk):
    ax.text(i, a, "%.1fx" % a, ha="center", va="bottom", fontsize=8)
ax.set_xticks(xpos)
ax.set_xticklabels(vers)
ax.set_yscale("log")
ax.set_ylabel("Speedup over CPU baseline")
ax.set_title("Lab 3: ReduceSum - NPU kernel speedup vs single-thread CPU")
ax.grid(axis="y", alpha=0.3, which="both")
ax.legend(frameon=False, loc="upper left")
for s in ("top", "right"):
    ax.spines[s].set_visible(False)
plt.tight_layout()
plt.show()

In [ ]:
sp_v1 = [base_v1 / r["kernel_ms"] for r in rows_main]
colors = [C_BASE] + [C_OPT] * (len(vers) - 1)

fig, ax = plt.subplots(figsize=(6.8, 4.0), dpi=120)
ax.bar(vers, sp_v1, 0.55, color=colors)
for i, v in enumerate(sp_v1):
    ax.text(i, v, "%.2fx" % v, ha="center", va="bottom", fontsize=9)
ax.set_ylabel("Speedup vs. v1 (single core, tile-wise reduce)")
ax.set_title("Lab 3: contribution of each optimization step")
ax.grid(axis="y", alpha=0.3)
for s in ("top", "right"):
    ax.spines[s].set_visible(False)
plt.tight_layout()
plt.show()

## 13. 参数扫描

上述四个版本回答的问题是：某一项优化该不该采用。本节考察两个连续参数的取值影响。修改核数不需要修改源码：直接在 `bisheng` 命令后追加 `-D` 即可；改变元素总数则连重新编译都不需要，直接作为命令行参数传入。

### 13.1 核数扫描：两种核间合并方式的对照

固定分块长度，改变参与计算的核数，同时记录 v3 与 v4 的耗时。二者的核内计算完全相同，差别只在核间合并方式，因此它们的差值就是**核间合并开销的差值**。

核数增加时，v3 第二阶段要合并的元素也随之增加，而 v4 的原子操作次数同样等于核数。两种方式的差距如何随核数变化，是本节要观察的问题。读右图时要先看纵轴的跨度：若两条曲线几乎重合、比值始终贴近 1，说明合并开销在总耗时中占比很小，此时不应把微小的差值当作趋势来解读。

> **读输出时的两点提醒。** 其一，每一档都会重新编译并运行一遍完整程序，因此 `cpu_ms` 每档都是重新测量的；输出行把它与加速比一并打印，正是为了让读者能够判断「v4 vs CPU」的变化究竟来自 NPU 侧还是基准侧（见 §15 ⑤）。考察核数扩展性应当看 `kernel_ms` 本身，也就是左图。其二，若扫描的核数超过设备实际可用的矢量核数，曲线会走平甚至回落，这与访存瓶颈是两回事，作图前应先确认可用核数。

In [ ]:
_built = {}


def build_and_run(defines=None, args=(), tag="scan"):
    # 与实验二同名：按给定的 -D 宏编译并运行，返回 (stdout, 是否成功)。
    # 与实验二的差别只有一处：同一个 tag 只编译一次，
    # 便于 §14 用同一个可执行文件跑多组运行时参数
    exe = "src_reduce/ascendc_reduce_sum_%s" % tag
    if tag not in _built:
        cmd = ["bisheng", SRC, "--npu-arch=" + ARCH, "-O2", "-o", exe]
        for k, v in (defines or {}).items():
            cmd.append("-D%s=%s" % (k, v))
        b = subprocess.run(cmd, capture_output=True, text=True)
        _built[tag] = b.returncode == 0
        if not _built[tag]:
            print("❌ 编译失败（%s）：%s" % (tag, (b.stdout + b.stderr).strip()[-300:]))
    if not _built[tag]:
        return "", False
    r = subprocess.run(
        ["./" + exe] + [str(a) for a in args],
        capture_output=True,
        text=True,
        timeout=900,
    )
    return r.stdout, ("[PERF]" in r.stdout)


block_dims = [1, 2, 4, 8, 16, 32]
scan_core = []

for bd in block_dims:
    txt, ok = build_and_run({"LAB_BLOCK_DIM": bd}, tag="bd%d" % bd)
    if not ok:
        print("❌ blockDim=%-3d 编译或运行失败（可能超出设备可用核数）" % bd)
        continue
    p = {r["ver"]: r for r in parse_perf(txt)}
    scan_core.append(
        (bd, p["v3"]["kernel_ms"], p["v4"]["kernel_ms"], p["v4"]["sp_kernel"])
    )
    print(
        "blockDim=%-3d v3=%.4f ms  v4=%.4f ms  v3/v4=%.3fx（>1 表示 v4 更快）"
        "  cpu=%.4f ms  v4 vs CPU=%.2fx"
        % (
            bd,
            p["v3"]["kernel_ms"],
            p["v4"]["kernel_ms"],
            p["v3"]["kernel_ms"] / p["v4"]["kernel_ms"],
            # 每一档都会重新测量一次 CPU 基准，因此必须与加速比一并输出，
            # 否则无法判断「v4 vs CPU」的变化来自 NPU 侧还是基准侧（见 §15 ⑤）
            p["v4"]["cpu_ms"],
            p["v4"]["sp_kernel"],
        )
    )

In [ ]:
if scan_core:
    bd = [s[0] for s in scan_core]
    ms3 = [s[1] for s in scan_core]
    ms4 = [s[2] for s in scan_core]
    sp3 = [ms3[0] / m for m in ms3]
    sp4 = [ms4[0] / m for m in ms4]

    fig, axes = plt.subplots(1, 2, figsize=(11.4, 4.2), dpi=120)

    ax = axes[0]
    ax.plot(bd, sp3, marker="o", lw=2, color=C_KERNEL, label="v3 workspace")
    ax.plot(bd, sp4, marker="s", lw=2, color=C_ALT, label="v4 atomic add")
    ax.plot(bd, bd, lw=1, ls="--", color="#bbbbbb", label="ideal linear")
    ax.set_xscale("log", base=2)
    ax.set_xticks(bd)
    ax.set_xticklabels(bd)
    ax.set_xlabel("blockDim (number of vector cores)")
    ax.set_ylabel("Speedup vs. blockDim = 1")
    ax.set_title("Core-count scaling")
    ax.grid(alpha=0.3)
    ax.legend(frameon=False, loc="upper left")

    ax = axes[1]
    ratio = [a / b for a, b in zip(ms3, ms4)]
    ax.plot(bd, ratio, marker="D", lw=2, color=C_OPT)
    ax.axhline(1.0, color="#888888", lw=1.0, ls=":")
    ax.set_xscale("log", base=2)
    ax.set_xticks(bd)
    ax.set_xticklabels(bd)
    ax.set_xlabel("blockDim (number of vector cores)")
    ax.set_ylabel("v3 time / v4 time  ( >1 means v4 faster )")
    ax.set_title("Atomic add vs. workspace two-stage")
    ax.grid(alpha=0.3)

    for a in axes:
        for s in ("top", "right"):
            a.spines[s].set_visible(False)
    plt.tight_layout()
    plt.show()

### 13.2 规模扫描：核间合并开销的占比

元素总数是运行时参数，这一组扫描不需要重新编译。

核间合并的开销**不随数据量增长而摊薄**：无论 N 是多少，v3 都要多启动一次核函数、多走一趟 workspace，v4 都要做 `BLOCK_DIM` 次原子操作。而第一阶段的耗时与 N 成正比。因此 N 越小，合并开销的占比越高；小到一定程度，多核版本相对单核版本的优势就会被合并开销吃掉。

扫描的取值必须是 `BLOCK_DIM × TILE_LENGTH = 32768` 的整数倍——这正是 `main` 开头那条检查所要求的。

扫描之后的单元格把每一档的 `kernel_ms` 换算成核函数侧读取输入的等效带宽，§15 会用它判断多核扩展受何种因素限制。

In [ ]:
sizes = [32768, 131072, 524288, 2097152, 8388608]
scan_n = []

print(
    "%10s %10s %11s %11s %11s %11s"
    % ("N", "CPU(ms)", "v2 kernel", "v3 kernel", "v4 kernel", "v4 vs CPU")
)
print("-" * 70)
for nn in sizes:
    txt = run_demo([nn])
    p = {r["ver"]: r for r in parse_perf(txt)}
    if not p:
        print("❌ N=%d 运行失败" % nn)
        continue
    scan_n.append((nn, p))
    print(
        "%10d %10.4f %11.4f %11.4f %11.4f %10.2fx"
        % (
            nn,
            p["v2"]["cpu_ms"],
            p["v2"]["kernel_ms"],
            p["v3"]["kernel_ms"],
            p["v4"]["kernel_ms"],
            p["v4"]["sp_kernel"],
        )
    )

In [ ]:
if scan_n:
    N = [s[0] for s in scan_n]
    sp2 = [s[1]["v2"]["sp_kernel"] for s in scan_n]
    sp3 = [s[1]["v3"]["sp_kernel"] for s in scan_n]
    sp4 = [s[1]["v4"]["sp_kernel"] for s in scan_n]

    fig, ax = plt.subplots(figsize=(7.6, 4.4), dpi=120)
    ax.plot(N, sp2, marker="o", lw=2, color=C_BASE, label="v2 single core (kernel)")
    ax.plot(N, sp3, marker="^", lw=2, color=C_KERNEL, label="v3 workspace (kernel)")
    ax.plot(N, sp4, marker="s", lw=2, color=C_OPT, label="v4 atomic add (kernel)")
    ax.axhline(1.0, color="#888888", lw=1.0, ls=":")
    ax.text(N[0], 1.06, "baseline: CPU = 1.0x", fontsize=9, color="#777777")
    ax.set_xscale("log", base=2)
    ax.set_yscale("log")
    ax.set_xlabel("Problem size N (elements)")
    ax.set_ylabel("Speedup over CPU baseline")
    ax.set_title("Lab 3: kernel speedup vs. problem size")
    ax.grid(alpha=0.3, which="both")
    ax.legend(frameon=False, loc="upper left", fontsize=9)
    for s in ("top", "right"):
        ax.spines[s].set_visible(False)
    plt.tight_layout()
    plt.show()

In [ ]:
# 把 §13.2 的 kernel_ms 换算成核函数侧读取输入的等效带宽：
#   等效带宽 = 4N / kernel_ms（每个 float 4 字节；规约的输出只有 32 字节，可忽略）
# 用途见 §15：判断多核扩展是否已受访存带宽限制，须看这个数值随 N 增大后是否停在
# 同一水平上，并把该数值与所用平台的标称带宽相比较——仅凭加速比曲线变平不足以下结论。


def eff_bw_gbs(n_elem, ms):
    return n_elem * 4 / ms / 1e6  # 元素数 × 4 字节 ÷ 毫秒 → GB/s


if scan_n:
    print("%10s %10s %12s %12s %12s" % ("N", "输入(MiB)", "v2 GB/s", "v3 GB/s", "v4 GB/s"))
    print("-" * 60)
    for nn, p in scan_n:
        print(
            "%10d %10.2f %12.1f %12.1f %12.1f"
            % (
                nn,
                nn * 4 / 1024 / 1024,
                eff_bw_gbs(nn, p["v2"]["kernel_ms"]),
                eff_bw_gbs(nn, p["v3"]["kernel_ms"]),
                eff_bw_gbs(nn, p["v4"]["kernel_ms"]),
            )
        )
    print()
    print("v2 为单核，v3 与 v4 为多核。若多核两列随 N 增大后趋于同一个数值，")
    print("说明存在一个与 N 无关的瓶颈；要断定该瓶颈就是 Global Memory 的读带宽，")
    print("还须把这个数值与所用平台的标称带宽相比较。")


## 14. 精度分析：验证串行累加链这条规律

§5 给出的规律是：相对误差大致按 ε·√k / √L 变化，k 为单条通道的串行链长，L 为独立通道数。本节用实测数据检验它。

单次测量的误差带有很强的随机性——同一种实现换一组数据，误差可能相差一个量级。因此这里对每个规模取**三组不同的随机种子**，报告中位数。数据种子是运行时参数，改变它不需要重新编译。

为缩短扫描时间，这里用 `-DLAB_REPEAT=1` 单独编译一份只跑一遍的可执行程序：精度分析不需要性能数据。

In [ ]:
prec_sizes = [32768, 131072, 524288, 2097152, 8388608]
prec_seeds = [2026, 20260711, 987654321]
VERS = ["cpu", "v1", "v2", "v3", "v4"]
prec = {v: [] for v in VERS}
chains = {}

print("%10s %11s %11s %11s %11s %11s" % ("N", "CPU f32", "v1", "v2", "v3", "v4"))
print("-" * 70)
for nn in prec_sizes:
    got = {v: [] for v in VERS}
    for sd in prec_seeds:
        # 同一个 tag 只在第一次调用时编译，后续直接复用该可执行文件
        txt, ok = build_and_run({"LAB_REPEAT": 1}, args=[nn, sd], tag="prec")
        if not ok:
            continue
        b = parse_base(txt)
        got["cpu"].append(b["cpu_rel_err"])
        for r in parse_rows(txt, "VERIFY"):
            got[r["ver"]].append(r["rel_err"])
            chains[r["ver"]] = (r["chain"], r["lanes"])
        chains["cpu"] = (b["chain"], 1)
    for v in VERS:
        prec[v].append(float(np.median(got[v])) if got[v] else float("nan"))
    print(
        "%10d %11.3e %11.3e %11.3e %11.3e %11.3e"
        % (
            nn,
            prec["cpu"][-1],
            prec["v1"][-1],
            prec["v2"][-1],
            prec["v3"][-1],
            prec["v4"][-1],
        )
    )

print()
print("N = %d 时各实现的累加结构：" % prec_sizes[-1])
print("%-6s %14s %14s" % ("实现", "链长 k", "通道数 L"))
for v in VERS:
    print("%-6s %14d %14d" % (v, chains[v][0], chains[v][1]))

In [ ]:
LBL = [
    ("cpu", "CPU f32 sequential", "#C7000B", "o"),
    ("v1", "v1 tile-wise reduce", "#E07A3B", "s"),
    ("v2", "v2 vector accumulator", "#3B6FE0", "^"),
    ("v4", "v4 multi-core + atomic", "#2E9E6B", "D"),
]

fig, ax = plt.subplots(figsize=(7.8, 4.6), dpi=120)
for key, label, color, mk in LBL:
    ax.plot(prec_sizes, prec[key], marker=mk, lw=2, color=color, label=label)
ax.axhline(1e-4, ls="--", color="black", lw=1.0, alpha=0.7)
ax.text(prec_sizes[0], 1.15e-4, "tolerance = 1e-4", fontsize=8)
ax.set_xscale("log", base=2)
ax.set_yscale("log")
ax.set_xlabel("Number of elements N")
ax.set_ylabel("Relative error (median of 3 seeds)")
ax.set_title("Lab 3: reduction error vs. problem size")
ax.grid(ls=":", alpha=0.5, which="both")
ax.legend(fontsize=9, frameon=False)
for s in ("top", "right"):
    ax.spines[s].set_visible(False)
plt.tight_layout()
plt.show()

## 15. 结果分析

> 以下结论针对**趋势规律**。具体数值随硬件规格、CANN 版本与系统负载而变化，请以本机的运行输出为准。

**① v1 → v2：减少规约次数**

v1 每个分块调用一次 `ReduceSum`，N = 2^21 时共 512 次；v2 只调用 1 次，其余全部替换为逐元素 `Add`。两处代价被同时省去：官方给出的实测口径是归约指令的延迟约为 `Add` 指令的 2 至 5 倍，而 `ReduceSum` 又由多条指令组合实现，其代价高于同长度的逐元素运算；每次规约之后的 `GetValue` 还会在矢量流水与标量流水之间引入一次依赖，打断流水的连续性。§3.3 给出的官方方案次序针对的正是前一项。

结果表中的 `规约次数` 一列就是这一改动的直接度量。**这与第三章 GEMV 的结论是同一条**：规约是向量化的难点，应当尽可能推迟到最后一步执行，中间过程让部分和保留在向量中。

**② v2 → v3：多核带来的加速**

这一步同时改变了并行度，其性质与前两个实验中「单核 → 多核」的那一步相同，因此通常是加速比提升最大的一步。但规约算子有一处不同：核间合并的开销**不随数据量增长而摊薄**。§13.2 的规模扫描正是为了定位这一开销的作用区间。把该表中 v2 与 v4 的 `kernel_ms` 相除即可看清：这个比值随规模单调上升，而在最小的一档上两者基本持平——此时数据太少，8 个核分到的工作量还不足以抵消多核启动与核间合并的固定开销。**多核不是无条件划算的，它有一个起效规模。**

**③ v3 → v4：两种核间合并方式的性能差别很小，原因不止一层**

v3 的第二阶段只处理 `BLOCK_DIM × 8` 个元素，计算量微不足道，耗时几乎全部来自核函数启动与 workspace 的读写；v4 用硬件原子累加省去了这两项。按这个推理，v4 应当更快。

**实测通常会发现两者的差别落在测量抖动之内。** §13.1 右图的 `v3 / v4` 耗时之比在 1.0 附近小幅波动，并且随核数**没有单调趋势**——某些核数下 v4 略快，另一些核数下反而是 v3 略快。原因有两层：

- **核内计算是主导项**。它在两个版本中逐字相同，占了总耗时的绝大部分，第二次核函数启动的增量开销相对它小到可以忽略。
- **v4 省下的那一趟 workspace 往返，并没有省下同地址串行化**。§4.1 已经指出，`BLOCK_DIM` = 8 时整个 workspace 只有 256 字节，与 v4 各核争用的那 32 字节同样落在一个 512 字节窗口之内。两个版本在这一点上受**完全相同**的硬件约束，而这项约束随核数增大而加剧——这也解释了为何在较大的核数上，v4 有时反而慢于 v3。

这个结果本身值得记住：**省掉一次核函数启动确实是一项优化，但它的绝对收益取决于被省掉的那部分在总耗时中占多大比重。** 当核内计算是主导项时，这类改动测不出差别。要让差别显现，需要把核内计算压到与合并开销同量级——例如把 N 减小若干个数量级，或在同一个 Stream 上反复调用该算子后取平均。

既然性能上没有明显差别，选择依据就落回工程因素：

<!-- markdown 版本（保留备用；如需切回，删除本注释标记并注释掉下方 HTML 表格）
| 对比项 | v3 workspace 两阶段 | v4 原子累加 |
| --- | --- | --- |
| 代码量 | 多一个算子类、多一次启动 | 更短 |
| 额外设备内存 | 需要 workspace | 不需要 |
| 对调用方的要求 | 无 | **每次启动前必须清零输出** |
| 泛化能力 | 第二阶段可做任意合并（加权、求最大值、多输出） | 只支持硬件提供的原子操作 |
| 核数很大时 | 第二阶段的元素数随核数增长；各核写 workspace 仍落在同一 512 字节窗口内 | 争用同一段 32 字节的核数随之增长，硬件串行化随之加剧 |
-->

<table style="margin-left:0; margin-right:auto; border-collapse:collapse; text-align:left;">
<thead>
<tr>
<th style="border:1px solid #cccccc; padding:6px 12px; text-align:left; background-color:#f5f5f5;">对比项</th>
<th style="border:1px solid #cccccc; padding:6px 12px; text-align:left; background-color:#f5f5f5;">v3 workspace 两阶段</th>
<th style="border:1px solid #cccccc; padding:6px 12px; text-align:left; background-color:#f5f5f5;">v4 原子累加</th>
</tr>
</thead>
<tbody>
<tr>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">代码量</td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">多一个算子类、多一次启动</td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">更短</td>
</tr>
<tr>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">额外设备内存</td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">需要 workspace</td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">不需要</td>
</tr>
<tr>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">对调用方的要求</td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">无</td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;"><strong>每次启动前必须清零输出</strong></td>
</tr>
<tr>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">泛化能力</td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">第二阶段可做任意合并（加权、求最大值、多输出）</td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">只支持硬件提供的原子操作</td>
</tr>
<tr>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">核数很大时</td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">第二阶段的元素数随核数增长；各核写 workspace 仍落在同一 512 字节窗口内</td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">争用同一段 32 字节的核数随之增长，硬件串行化随之加剧</td>
</tr>
</tbody>
</table>

**核数扩展的形状**（§13.1 左图）：两个版本都明显偏离理想线性，且每把核数翻一倍所带来的改善持续衰减。偏离线性可能来自三个并存的原因，判别它们各自需要一组对照：

<!-- markdown 版本（保留备用；如需切回，删除本注释标记并注释掉下方 HTML 表格）
| 可能的成因 | 判别方式 |
| --- | --- |
| 启动与调度的固定开销 | 减小 N 后，偏离线性在更低的核数处即已出现 |
| Global Memory 读带宽饱和 | §13.2 之后那个单元格给出的等效带宽随 N 增大后停在同一数值上，且该数值接近平台的标称带宽 |
| 核间合并处的同地址串行化 | 按动手练习第 6 题把 workspace 槽位撑开到 512 字节后，高核数档位出现改善 |
-->

<table style="margin-left:0; margin-right:auto; border-collapse:collapse; text-align:left;">
<thead>
<tr>
<th style="border:1px solid #cccccc; padding:6px 12px; text-align:left; background-color:#f5f5f5;">可能的成因</th>
<th style="border:1px solid #cccccc; padding:6px 12px; text-align:left; background-color:#f5f5f5;">判别方式</th>
</tr>
</thead>
<tbody>
<tr>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">启动与调度的固定开销</td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">减小 N 后，偏离线性在更低的核数处即已出现</td>
</tr>
<tr>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">Global Memory 读带宽饱和</td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">§13.2 之后那个单元格给出的等效带宽随 N 增大后停在同一数值上，且该数值接近平台的标称带宽</td>
</tr>
<tr>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">核间合并处的同地址串行化</td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">按动手练习第 6 题把 workspace 槽位撑开到 512 字节后，高核数档位出现改善</td>
</tr>
</tbody>
</table>

这里有一处容易走捷径的地方：**等效带宽停在某个数值上，只说明存在一个与数据量无关的瓶颈，并不足以断定这个瓶颈就是访存带宽。**要下这个结论，还须把实测的等效带宽与所用平台的标称带宽相比较；若二者相差一个量级，限制因素多半在别处。这与实验二核数扫描的判别方法是同一套做法。

需要强调的是，**v4 之所以不比 v3 慢，前提同样是原子操作的次数很少**（等于核数）。若把 v4 改成「每个分块规约后就原子累加一次」，原子操作次数将上升到分块数，性能会急剧恶化——那时的情形就与第五章直方图实验的 V2 相同了。动手练习第 2 题要求实际测出这个反例。

同时也要记住 §4.2 指出的隐性成本：原子累加要求**调用方**在每次启动前清零输出。在一个反复调用该算子的循环中，这项成本是要计入的。

**④ CPU 对比：为什么不以「主机往返一次」为口径**

本实验的加速比一律以核函数耗时为准，不测量「主机拷贝 + 核函数 + 设备回传」这一端到端口径，理由与实验二 §5.1 相同：真实计算图中，数据一次搬入设备内存后会常驻其上，由连续的多个算子接力处理，直至整段计算结束才回传一次结果；为单个算子各安排一次完整往返的调用方式在工程上并不存在，据此算出的比值也回答不了「该算子是否值得放到 NPU 上执行」——这一判断的单位是整段计算图，而不是单个算子。

就本算子而言，从输入输出的规模即可作出判断：输入为 4N 字节，输出只有 32 字节。若为它单独安排一次主机与设备之间的往返，需要搬运的字节数与需要计算的元素数同阶，而 CPU 侧完成同一件事也只需把同样的数据读一遍。按第一章的 **Amdahl 定律**，数据搬运是一段无法被加速的前缀，它在总时间中的占比决定了整体收益的上限；对这一类算子而言，这个占比很高。

> **工程结论**：不宜为单个算子在主机与设备之间往返搬运数据。真实的推理框架让数据一次搬入后常驻设备内存，由连续的多个算子接力处理，并进一步把相邻算子**融合**为一个核函数。规约在其中通常也不是终点，它的输出往往立刻被下一个算子消费，根本不回传主机。这正是实验五算子融合要讨论的问题。至于主机与设备之间这条数据通路本身的性质——锁页内存、异步拷贝与多 Stream 重叠——属于应用开发的范畴，第七章会系统讨论。

**⑤ CPU 基准的性质：它同时是最慢的和最不准的**

本实验的 CPU 基准是单线程 `float32` 顺序累加，它有两个特点，且出自同一个原因——**串行累加链的长度等于 N**：

- 性能上，2^21 次浮点加法构成一条完整的依赖链，每一次加法都必须等待上一次的结果，无法利用 CPU 的多发射与向量化。这不是一个经过优化的基准。
- 精度上，它是全部五种实现中误差最大的一个，§14 的图中它应当明显位于最上方。

因此本实验的加速比数字包含两部分：一部分来自 NPU，另一部分来自基准实现本身的朴素。动手练习第 3 题要求把 CPU 基准改成多累加器展开的版本，量化后者占多少——这与第三章、第五章中打断累加链的循环依赖是同一个话题。

这正是实验二 §13 ⑤ 那条规范在本实验中的具体表现：**加速比由分子与分母共同决定，脱离基准耗时的加速比没有意义**。§13.1 的核数扫描每一档都会重新测量一次 CPU 基准，因此该输出中的「v4 vs CPU」同时受核数与基准波动两个因素影响，不能单独用作核数扩展性的证据；判断扩展性应当看 `kernel_ms` 本身，也就是左图的曲线。本实验的 `[PERF]` 记录行同时输出 `cpu_ms` 与 `kernel_ms` 两个原始耗时，正是为了让读者能够还原比值的来源。

**⑥ 精度：并行版本比串行版本更准**

读 §14 的图时有三点需要确认：

- **CPU 串行版本应当明显位于最上方**，与四个 NPU 版本相差若干个数量级。这一条由 §5 的规律直接推出，且差距足够大，不会被随机性淹没。
- **CPU 曲线应当总体随 N 上升**，因为它的链长就等于 N。每个规模只取了三组种子，中位数仍有波动，个别档位出现回落属于正常；四个 NPU 版本的链长只有 N 的几百分之一到几万分之一，误差随 N 的变化要平缓得多，在本实验的规模范围内甚至看不出趋势。
- **四个 NPU 版本之间的排序未必与 §5 的表格逐一吻合**。它们的误差都在 1e-9 到 1e-7 之间，彼此相差不到两个数量级，而 §5 的公式只是数量级估计；更重要的是，`ReduceSum` 内部的折半结构会额外贡献一部分误差，这部分没有计入 (k, L) 两个量。因此这里能可靠观察到的是**串行链版本与向量累加版本之间的分界**，而不是四个 NPU 版本的精确次序。

**四个 NPU 版本的误差已经接近 float32 的分辨极限。** 按本实验的数据规格核算：输入取自 [0, 1) 上的均匀分布，N = 2^21 时求和结果约为 N/2 ≈ 1.05×10^6，落在 [2^19, 2^20) 这个二进制区间内，float32 在此处的最小间隔（1 ULP）为 0.0625，换算成相对量约 6×10^-8。四个 NPU 版本的相对误差都落在个位数个 ULP 的范围内，而 CPU 串行版本要高出两个数量级以上。**误差落到个位数 ULP 时，实现之间的排序已经没有意义**，那是舍入的随机性而不是算法优劣。这也正是上一条的原因：§5 的公式在这个量级上失去了分辨力。

这一结果与常见的直觉相反。「并行化改变了累加顺序，所以结果不如串行版本可靠」这句话只说对了前半句：顺序确实改变了，但改变之后误差是变大还是变小，取决于**链变长了还是变短了**。分块累加把一条长链拆成许多条短链，误差因此下降——这也正是 NumPy 的 `sum` 采用成对累加（pairwise summation）而不是朴素顺序累加的原因。

由此得到本实验对容差的处理方式：**容差应当从算法的误差来源推出**。本实验取 1e-4，覆盖的是误差最大的那个实现（CPU 串行）；四个 NPU 版本的实际误差要小若干个数量级。若某个版本的误差接近容差，那说明的不是「容差定得太严」，而是该实现的累加链出了问题。



---

### 🎓 结论

规约揭示了并行算子的第三个核心问题：**数据依赖的合并**。其结论可以概括为两条——

**核内层面**：让部分和保留在向量中，把规约推迟到最后一步执行；

**核间层面**：先在核内把数据压缩为极少量的标量，再选择合并方式。只要原子操作的次数被压到与核数同量级，原子累加就是最简洁的合并手段。

而贯穿两个层面的，是同一个量：**串行累加链的长度**。缩短它同时改善性能与精度——在规约这一类算子上，快与准并不冲突。

## 16. 🔧 动手练习

> **提示**：修改源码需要从 §9.1 开始按顺序重新执行全部写入单元格。只改参数的练习用 `build_and_run({...}, tag='...')` 一行即可完成；只改规模或种子的练习连重新编译都不需要，`run_demo([n, seed])` 即可。

1. **制造一次清零错误**。把 `KernelReduceVecAccum::Process` 中的 `Duplicate(acc, 0, TILE_LENGTH)` 移到 `for` 循环内部，重新编译运行。由于 v2、v3、v4 共用这个类，三者会同时出错，而 v1 不受影响——这本身就是共用算子类的一个副作用。请记录 v2 得到的数值并解释它等于什么，再解释为什么 v3 与 v4 得到的数值约为 v2 的 8 倍。最后回答：容差取 1e-4 时，这个错误能否被拦住？

2. **把原子累加移进循环**。把 v4 改成「每个分块规约后立即原子累加一次」，即把 `ReduceSum` 与原子搬出一并移入分块循环。记录性能变化的倍数，并与第五章直方图实验 V2 的结论对照，说明两者为何一致。

3. **改进 CPU 基准**。把 `RunOnCpu` 改为四累加器展开（`s0 += x[i]; s1 += x[i+1]; ...`，末尾再把四个累加器相加），重新测量 `cpu_ms` 与 `cpu_rel_err`。加速比下降了多少？误差下降了多少？请用 §5 的 (k, L) 两个量解释这两个变化。

4. **合并开销的作用区间**。用 `run_demo([n])` 在 32768、65536、131072、262144 四个规模上运行，比较 v2 与 v4 的 `kernel_ms`。v4 从哪个规模开始明显快于 v2？请从核间合并开销的占比解释。*注意：取值必须是 `BLOCK_DIM × TILE_LENGTH = 32768` 的整数倍，否则程序会直接报错退出——这条约束在 §10.2 中有说明。*

5. **分块长度与通道数**。用 `build_and_run({'LAB_TILE_LENGTH': tl}, tag='tl%d' % tl)` 依次取 1024、2048、4096、8192，同时记录 v2 的 `kernel_ms` 与 `rel_err`。分块长度同时是向量累加器的通道数 L，因此它对性能和精度都有影响，且方向未必一致。请分别作图并说明。

6. **撑开 workspace 槽位，规避同地址串行化**。把 `WS_BLOCK` 由 `ALIGN_ELEM`（8 个元素、32 字节）改为 128 个元素（512 字节），使各核写入的位置落入不同的 512 字节窗口。`WS_LENGTH`、`WS_BYTES` 与第二阶段的规约长度都会随之改变，请先核算第二阶段 `ReduceSum` 的临时缓冲是否仍然够用，再动手。改完后在 `blockDim` 取 8、16、32 三档上重测 v3，与改动前对照。这道题检验的是 §4.1 给出的机制：**若改善主要出现在高核数档位，同地址串行化的解释即成立**。顺带说明为什么这个改动对 v4 无效。

7. **【进阶】用 `SyncAll` 取代两次启动**。查阅 `AscendC::SyncAll` 的用法（它需要一块 `int32_t` 类型的全局缓存保存各核的状态标记，该缓存同样由 workspace 提供），把 v3 改写为单次启动：各核写完 workspace 后集体等待，再由 0 号核完成合并。比较改写前后的耗时，并说明这种写法在什么场景下更有优势。*注意 §4.1 提到的约束：使用核间同步时逻辑核数不得大于设备实际可用的核数，否则核函数会卡死；因此这一版本不能沿用 §13.1 中最大的那几档核数。*

8. **【进阶】按官方的二分累加方案改写末尾的规约**。参照 §3.3，用 `WholeReduceSum`，或「反复折半 `Add` + `WholeReduceSum`」替代 v2 末尾的 `ReduceSum`，比较性能差异。**请分别在 v1 与 v2 上做这个改动**：v1 要执行 512 次规约，v2 只执行 1 次，官方给出的方案次序应当在前者上体现得远比后者明显。请先据此预估两个版本各自的收益，再用实测检验预估是否成立。

## 17. 🤔 思考题

1. 为什么规约必须分成核内与核间两级？能否设计一种只有一级的实现？如果可以，其代价是什么？

2. v3 的 workspace 中，为何每个核要占用 8 个元素而不是 1 个？如果改成每个核只写 1 个元素，会出现什么问题？

3. v3 用两次核函数启动保证阶段顺序，另一种做法是在单次启动内使用 `SyncAll`。请对比二者的开销与适用场景，并说明为什么只有后者对启动的核数有硬性上限。

4. v4 若忘记调用 `SetAtomicNone`，会在什么时候、以什么形式表现出错误？这类错误为何特别难以定位？

5. `GetValue` 属于 Scalar 流水操作。在关闭自动同步的工程中，它与前一条 `ReduceSum` 之间应当插入哪一类同步事件？为什么方向是 `V_S` 而不是 `S_V`？

6. 第五章的结论是「原子操作代价高」，本实验的结论是「原子累加是最简洁的合并方式」。这两条结论是否矛盾？判断二者适用性的**同一个判据**是什么？

7. 若把输入数据由 [0, 1) 改为 [−1, 1)，求和结果将落在 0 附近。此时相对误差判据会出现什么问题？应当改用何种判据？

8. §5 给出的误差规律是 ε·√k / √L。按这个规律，把 `TILE_LENGTH` 增大一倍，v2 的误差应当变为原来的几分之几？请先推算，再用动手练习第 5 题的数据检验。

9. CPU 基准的累加器初值取自一个 `volatile` 变量，目的是防止编译器把整个重复循环提走。如果不这样做，测得的 `cpu_ms` 会接近于什么值？由此得到的加速比会是什么样子？这类「测量代码被优化掉」的错误，为什么在标量输出的算子上特别容易发生？

10. §15 ③ 显示 v3 与 v4 的耗时差别落在测量抖动之内。如果要设计一个能把这个差别测出来的实验，应当怎么改？请给出至少两种思路，并说明各自会引入什么新的干扰因素。

11. 假设一个应用需要对同一份数据连续做三次规约（每次用不同的权重）。按 §15 ④ 的分析，应当如何组织这三次计算，才能使主机与设备之间的搬运总量最小？请分别估算「每次都完整往返一遍」与「只在首尾各搬运一次」两种组织方式下的搬运总量之比。如果这三次规约之间还夹着一个逐元素乘法呢？

## 18. 📌 本实验小结

<!-- markdown 版本（保留备用；如需切回，删除本注释标记并注释掉下方 HTML 表格）
| 概念 | 要点 |
| --- | --- |
| 规约的数据依赖 | 唯一输出依赖全部输入，核间**必须**交换数据 |
| 两级规约 | 核内规约受片上缓冲约束，核间规约必须借道 Global Memory |
| 归约 API 家族 | `Reduce*` 归约全部数据、`WholeReduce*` 归约每个 repeat、`BlockReduce*` 归约每个 datablock、`PairReduceSum` 归约相邻两元素 |
| `ReduceSum` | 需额外的临时缓冲，所需元素数按 repeat 数向上对齐核算；由多条指令组合实现，不是单条硬件指令 |
| 官方的方案次序 | 大数据量、循环次数多的场景下：二分累加 > `WholeReduceSum` 单指令 > `ReduceSum` 接口 |
| `TBuf` 与 `TQue` | 判据是数据是否跨流水任务传递；`TBuf` 无缓冲块数、无同步语义 |
| 向量累加器 | 部分和保留在向量中，规约只在最后执行一次 |
| 累加器清零 | 必须在循环之前，且只能做一次 |
| workspace | 算子内部使用的设备内存，既非输入也非输出 |
| 两阶段规约 | 同一 Stream 内任务串行执行，两次启动即保证阶段顺序 |
| `SetAtomicAdd` | 使 `DataCopy` 由覆盖变为累加；启动前须清零，用完须 `SetAtomicNone` |
| 原子累加的隐性成本 | 清零输出的责任转移给调用方；结果必须在计时循环之前取走 |
| 原子操作的判据 | 代价取决于**总次数**，而非是否使用了原子指令 |
| 同地址串行化 | 多核访问落在同一连续 512 字节范围内的地址时，请求被硬件串行处理，核数越多劣化越严重 |
| 两种核间合并方式的取舍 | 性能差别很小，因为二者受同一条同地址串行化约束；选择依据是代码量、额外显存与泛化能力，而非速度 |
| 串行累加链 | 链长 k 与通道数 L 同时决定性能与精度，误差约按 ε·√k / √L 变化 |
| 浮点规约的误差 | 并行版本与串行版本不可能逐位相同，但并行版本通常**更准** |
| 精度校验 | 参考值的精度必须高于被测对象，故真值以 `float64` 计算 |
| 容差的选取 | 从算法的误差来源推出，而不是放大到「能通过为止」 |
| 标量输出的性能测量 | 仅使用结果还不够，还须防止重复循环被编译器提走 |
| 卸载收益的判断单位 | 是整段计算图而非单个算子；数据一次搬入后应尽可能长时间地留在 Device 上 |
-->

<table style="margin-left:0; margin-right:auto; border-collapse:collapse; text-align:left;">
<thead>
<tr>
<th style="border:1px solid #cccccc; padding:6px 12px; text-align:left; background-color:#f5f5f5;">概念</th>
<th style="border:1px solid #cccccc; padding:6px 12px; text-align:left; background-color:#f5f5f5;">要点</th>
</tr>
</thead>
<tbody>
<tr>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">规约的数据依赖</td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">唯一输出依赖全部输入，核间<strong>必须</strong>交换数据</td>
</tr>
<tr>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">两级规约</td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">核内规约受片上缓冲约束，核间规约必须借道 Global Memory</td>
</tr>
<tr>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">归约 API 家族</td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;"><code>Reduce*</code> 归约全部数据、<code>WholeReduce*</code> 归约每个 repeat、<code>BlockReduce*</code> 归约每个 datablock、<code>PairReduceSum</code> 归约相邻两元素</td>
</tr>
<tr>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;"><code>ReduceSum</code></td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">需额外的临时缓冲，所需元素数按 repeat 数向上对齐核算；由多条指令组合实现，不是单条硬件指令</td>
</tr>
<tr>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">官方的方案次序</td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">大数据量、循环次数多的场景下：二分累加 &gt; <code>WholeReduceSum</code> 单指令 &gt; <code>ReduceSum</code> 接口</td>
</tr>
<tr>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;"><code>TBuf</code> 与 <code>TQue</code></td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">判据是数据是否跨流水任务传递；<code>TBuf</code> 无缓冲块数、无同步语义</td>
</tr>
<tr>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">向量累加器</td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">部分和保留在向量中，规约只在最后执行一次</td>
</tr>
<tr>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">累加器清零</td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">必须在循环之前，且只能做一次</td>
</tr>
<tr>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">workspace</td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">算子内部使用的设备内存，既非输入也非输出</td>
</tr>
<tr>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">两阶段规约</td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">同一 Stream 内任务串行执行，两次启动即保证阶段顺序</td>
</tr>
<tr>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;"><code>SetAtomicAdd</code></td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">使 <code>DataCopy</code> 由覆盖变为累加；启动前须清零，用完须 <code>SetAtomicNone</code></td>
</tr>
<tr>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">原子累加的隐性成本</td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">清零输出的责任转移给调用方；结果必须在计时循环之前取走</td>
</tr>
<tr>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">原子操作的判据</td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">代价取决于<strong>总次数</strong>，而非是否使用了原子指令</td>
</tr>
<tr>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">同地址串行化</td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">多核访问落在同一连续 512 字节范围内的地址时，请求被硬件串行处理，核数越多劣化越严重</td>
</tr>
<tr>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">两种核间合并方式的取舍</td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">性能差别很小，因为二者受同一条同地址串行化约束；选择依据是代码量、额外显存与泛化能力，而非速度</td>
</tr>
<tr>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">串行累加链</td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">链长 k 与通道数 L 同时决定性能与精度，误差约按 ε·√k / √L 变化</td>
</tr>
<tr>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">浮点规约的误差</td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">并行版本与串行版本不可能逐位相同，但并行版本通常<strong>更准</strong></td>
</tr>
<tr>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">精度校验</td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">参考值的精度必须高于被测对象，故真值以 <code>float64</code> 计算</td>
</tr>
<tr>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">容差的选取</td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">从算法的误差来源推出，而不是放大到「能通过为止」</td>
</tr>
<tr>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">标量输出的性能测量</td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">仅使用结果还不够，还须防止重复循环被编译器提走</td>
</tr>
<tr>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">卸载收益的判断单位</td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">是整段计算图而非单个算子；数据一次搬入后应尽可能长时间地留在 Device 上</td>
</tr>
</tbody>
</table>

### 一条贯穿本章的原则

> **并行化的难度取决于数据依赖的形态，而非计算本身的复杂度。**

向量加法的计算是一次浮点加法，规约的计算也是浮点加法，但后者的实现复杂度高出一个层次——差别完全来自多对一这一依赖形态。

### 与后续实验的衔接

➡️ **后续内容：实验四 · Sigmoid**。本实验与实验二的计算都极为简单，一条基础 API 即可完成。下一个实验转向**复合函数**：Sigmoid 需要四条基础 API 串联，由此引出两个新问题——高阶 API 如何把这条链收敛为一次调用，以及当中间量超出数据类型的表示范围时会发生什么。